
# Experiment 22 — iTransformer + Frozen Historical Memory

## 연구 질문

Experiment 21에서 strong official-style iTransformer baseline을 고정했습니다.

이제 PatchTST에서 사용했던 historical-memory augmentation을 **구조 변경 없이**
iTransformer에 적용합니다.

핵심 검증은 다음과 같습니다.

$$
\boxed{
F_{\mathrm{iTransformer}}
+
R_{\mathrm{historical}}
>
F_{\mathrm{iTransformer}}
}
$$

대상은

$$
\mathcal{D}
=
\{
\mathrm{ETTm1},
\mathrm{ETTh1},
\mathrm{Weather},
\mathrm{Electricity}
\}
$$

와

$$
H
\in
\{
96,192,336,720
\}
$$

입니다.

총 16개 조건을 모두 수행합니다.

---

# Frozen augmentation

PatchTST Experiment 19와 동일한 method를 사용합니다.

$$
\boxed{
\text{Predictive Retrieval}
\rightarrow
\text{Uniform Top-10 Historical Forecast}
\rightarrow
\text{Cross-Fitted Adaptive Gate}
\rightarrow
\text{Validation-Calibrated Shrinkage}
}
$$

## 1. Predictive retrieval

Experiment 13에서 이미 학습한 EmbeddingOnly retriever를 그대로 사용합니다.

$$
s(q,i)
=
\gamma
\cos(
z_q,
z_i
)
$$

- retrieval lookback: \(L_R=96\)
- patch length: 16
- patch stride: 16
- embedding dimension: 64
- Top-\(K=10\)
- memory stride: 24
- retriever architecture: frozen
- retriever checkpoint: frozen
- retriever hyperparameters: frozen

## 2. Historical forecast

Top-10 candidate의 미래 residual을 단순 평균합니다.

$$
\hat{\mathbf y}_{\mathrm{hist}}
=
\frac{1}{K}
\sum_{i\in\mathcal{N}_K(q)}
\mathbf y_i
$$

새로운 set aggregator는 사용하지 않습니다.

## 3. Adaptive trust

$$
\hat{\mathbf y}
=
(1-\alpha_q)
\hat{\mathbf y}_{\mathrm{iTransformer}}
+
\alpha_q
\hat{\mathbf y}_{\mathrm{hist}}
$$

Gate 구조는 PatchTST 실험과 동일합니다.

$$
26
\rightarrow
64
\rightarrow
32
\rightarrow
1
$$

## 4. Validation-calibrated shrinkage

$$
\alpha_{\mathrm{final}}
=
(1-\lambda)\alpha_0
+
\lambda\alpha_{\mathrm{gate}}
$$

- \(\alpha_0\): validation-only scalar weight
- \(\lambda\): validation-only shrinkage coefficient

---

# Leakage-free cross-fitting

train split 내부에서 세 개의 chronological fold를 사용합니다.

$$
0.55\rightarrow0.70
$$

$$
0.70\rightarrow0.85
$$

$$
0.85\rightarrow1.00
$$

각 fold에서:

1. prefix 구간만으로 external standardization
2. prefix 구간만으로 iTransformer 학습
3. prefix 이전에 future까지 완전히 관측된 history만 retrieval memory로 사용
4. 바로 뒤 OOF 구간에서 direct / retrieval prediction 생성
5. OOF target은 adaptive gate 학습에만 사용

Fold iTransformer의 epoch 수는 OOF target으로 선택하지 않습니다.

Experiment 21 full iTransformer에서 validation으로 선택한 best epoch를 고정하여 사용합니다.

$$
E_{\mathrm{fold}}
=
E_{\mathrm{full}}^{*}
$$

---

# Validation

- full iTransformer: Experiment 21 checkpoint
- full retriever: Experiment 13 frozen checkpoint
- retrieval memory: train only
- scalar \(\alpha_0\): validation only
- gate epoch: validation only
- shrinkage \(\lambda\): validation only

# Test

- iTransformer: train-only Experiment 21 checkpoint
- retrieval memory: train + validation only
- no rolling test-label memory
- all valid test origins
- stride 1
- all channels

---

# Computational engineering

Experiment 19에서는 Electricity에서 direct backbone을 작은 retrieval batch마다 반복 호출하는 비용이 컸습니다.

이번 notebook은 scientific protocol을 바꾸지 않고 다음을 최적화합니다.

1. iTransformer direct prediction을 큰 anchor block으로 먼저 계산
2. 동일 block 내부에서 retrieval만 작은 sub-batch로 수행
3. fold checkpoint / OOF feature / memory embedding / validation feature를 즉시 cache
4. 완료된 dataset × horizon 조건은 자동 skip
5. kernel이 종료되어도 `RESUME=True`로 재개

따라서 Experiment 19보다 상당히 빠를 것으로 기대하지만,
Electricity의 321-channel retrieval과 3-fold cross-fitting 때문에 여전히 수 시간이 걸릴 수 있습니다.


In [1]:

from pathlib import Path
from types import SimpleNamespace
from contextlib import nullcontext

import gc
import importlib
import math
import random
import subprocess
import sys
import time
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 240)
pd.set_option("display.width", 500)

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

DATASETS = [
    "ETTm1",
    "ETTh1",
    "Weather",
    "Electricity",
]

HORIZONS = [
    96,
    192,
    336,
    720,
]

TASKS = [
    (dataset, horizon)
    for dataset in DATASETS
    for horizon in HORIZONS
]

DIRECT_SEQ_LEN = 96
LABEL_LEN = 48
RET_SEQ_LEN = 96

DIRECT_SEED = 2023
CROSSFIT_SEED = 2222

TOP_K = 10
MEMORY_STRIDE = 24
OOF_ANCHOR_STRIDE = 4

FOLDS = [
    (0.55, 0.70),
    (0.70, 0.85),
    (0.85, 1.00),
]

# Frozen predictive representation.
REP_PATCH_LEN = 16
REP_PATCH_STRIDE = 16
REP_D_MODEL = 64
REP_N_HEADS = 4
REP_LAYERS = 2
REP_D_FF = 128
REP_DIM = 64
REP_DROPOUT = 0.1

REP_NUM_PATCHES = (
    1
    + (
        RET_SEQ_LEN
        - REP_PATCH_LEN
    )
    // REP_PATCH_STRIDE
)

# Frozen adaptive-gate design.
GATE_DIM = 26
GATE_LR = 1e-3
GATE_WD = 1e-4
GATE_BATCH = 8192
GATE_MAX_EPOCHS = 50
GATE_PATIENCE = 7

ALPHA_GRID = np.round(
    np.arange(
        0.0,
        1.0001,
        0.1,
    ),
    10,
)

LAMBDA_GRID = np.array(
    [
        0.0,
        0.25,
        0.50,
        0.75,
        1.00,
    ],
    dtype=np.float32,
)

EPS = 1e-8
RETRIEVER_USE_AMP = torch.cuda.is_available()

# Small retrieval batches prevent GPU blow-up on 321-channel Electricity.
TARGET_RETRIEVAL_PAIRS = 672

# Direct iTransformer is evaluated in much larger anchor blocks.
DIRECT_ANCHOR_BLOCK = {
    "ETTm1": 128,
    "ETTh1": 128,
    "Weather": 64,
    "Electricity": 16,
}

EXP21_ROOT = Path(
    "/data/dataset/strong_forecaster/"
    "itransformer_official_baseline_reproduction"
)

FROZEN_RET_ROOT = Path(
    "/data/dataset/strong_forecaster/"
    "multidataset_crossfit_screening"
)

ROOT = Path(
    "/data/dataset/strong_forecaster/"
    "itransformer_plus_frozen_historical_memory"
)

DIRS = {
    "fold_direct": ROOT / "fold_direct",
    "oof": ROOT / "oof",
    "validation": ROOT / "validation",
    "memory_emb": ROOT / "memory_embeddings",
    "gate": ROOT / "gate",
    "history": ROOT / "history",
    "paired": ROOT / "paired_test",
    "channel": ROOT / "channel_test",
    "calibration": ROOT / "calibration",
    "artifacts": ROOT / "artifacts",
}

for p in DIRS.values():
    p.mkdir(
        parents=True,
        exist_ok=True,
    )

RESUME = True
FORCE = False

DIRECT_PARITY_TOL = 2e-5

print("Device:", DEVICE)
print("Tasks:", len(TASKS))
print("Experiment 21:", EXP21_ROOT)
print("Frozen retrievers:", FROZEN_RET_ROOT)
print("Output:", ROOT)


Device: cuda
Tasks: 16
Experiment 21: /data/dataset/strong_forecaster/itransformer_official_baseline_reproduction
Frozen retrievers: /data/dataset/strong_forecaster/multidataset_crossfit_screening
Output: /data/dataset/strong_forecaster/itransformer_plus_frozen_historical_memory


## 1. Load the exact official iTransformer implementation used in Experiment 21

In [2]:

REPO_CANDIDATES = [
    Path(
        "/code/stock_regime_retrieval/"
        "strong_forecaster/iTransformer_official"
    ),
    Path("/code/iTransformer"),
    Path("/data/iTransformer"),
]

REPO = next(
    (
        p
        for p in REPO_CANDIDATES
        if (
            p / "model" / "iTransformer.py"
        ).is_file()
    ),
    None,
)

if REPO is None:
    raise FileNotFoundError(
        "Official iTransformer repository was not found. "
        "Experiment 21 must be available before Experiment 22."
    )

for module_name in list(
    sys.modules.keys()
):
    if (
        module_name == "model"
        or module_name.startswith("model.")
        or module_name == "layers"
        or module_name.startswith("layers.")
        or module_name == "utils"
        or module_name.startswith("utils.")
    ):
        del sys.modules[
            module_name
        ]

if str(REPO) in sys.path:
    sys.path.remove(
        str(REPO)
    )

sys.path.insert(
    0,
    str(REPO),
)

itransformer_module = importlib.import_module(
    "model.iTransformer"
)

tools_module = importlib.import_module(
    "utils.tools"
)

timefeatures_module = importlib.import_module(
    "utils.timefeatures"
)

OfficialITransformer = (
    itransformer_module.Model
)

official_adjust_lr = (
    tools_module.adjust_learning_rate
)

official_time_features = (
    timefeatures_module.time_features
)

actual_model_file = Path(
    itransformer_module.__file__
).resolve()

expected_model_file = (
    REPO
    / "model"
    / "iTransformer.py"
).resolve()

if (
    actual_model_file
    != expected_model_file
):
    raise RuntimeError(
        "Wrong iTransformer implementation imported.\n"
        f"Expected: {expected_model_file}\n"
        f"Actual:   {actual_model_file}"
    )

try:
    commit = subprocess.check_output(
        [
            "git",
            "-C",
            str(REPO),
            "rev-parse",
            "HEAD",
        ],
        text=True,
    ).strip()
except Exception:
    commit = "unknown"

print(
    "PASS: official iTransformer implementation."
)
print("Repo:", REPO)
print("Commit:", commit)

(
    DIRS["artifacts"]
    / "official_repo_commit.txt"
).write_text(
    commit + "\n"
)


PASS: official iTransformer implementation.
Repo: /code/stock_regime_retrieval/strong_forecaster/iTransformer_official
Commit: c2426e68ca13f74aaec08045c5c724d8ad328124


41

## 2. Experiment 21 frozen direct checkpoints and recipes

In [3]:

EXP21_SUMMARY_PATH = (
    EXP21_ROOT
    / "summary.csv"
)

if not EXP21_SUMMARY_PATH.is_file():
    raise FileNotFoundError(
        f"Experiment 21 summary not found: "
        f"{EXP21_SUMMARY_PATH}"
    )

EXP21_SUMMARY = pd.read_csv(
    EXP21_SUMMARY_PATH
)

display(
    EXP21_SUMMARY[
        [
            "Dataset",
            "Horizon",
            "BestEpoch",
            "Test_MSE",
            "Test_MAE",
        ]
    ].sort_values(
        [
            "Dataset",
            "Horizon",
        ]
    )
)


RECIPES = {
    "ETTm1": {
        "data": "ETTm1",
        "enc_in": 7,
        "e_layers": 2,
        "d_model": 512,
        "d_ff": 2048,
        "n_heads": 8,
        "dropout": 0.1,
        "batch_size": 32,
        "learning_rate": 1e-4,
        "train_epochs": 10,
        "patience": 3,
        "lradj": "type1",
        "factor": 1,
        "freq": "t",
    },
    "ETTh1": {
        "data": "ETTh1",
        "enc_in": 7,
        "e_layers": 2,
        "d_model": 512,
        "d_ff": 2048,
        "n_heads": 8,
        "dropout": 0.1,
        "batch_size": 32,
        "learning_rate": 1e-4,
        "train_epochs": 10,
        "patience": 3,
        "lradj": "type1",
        "factor": 1,
        "freq": "h",
    },
    "Weather": {
        "data": "custom",
        "enc_in": 21,
        "e_layers": 3,
        "d_model": 512,
        "d_ff": 512,
        "n_heads": 8,
        "dropout": 0.1,
        "batch_size": 32,
        "learning_rate": 1e-4,
        "train_epochs": 10,
        "patience": 3,
        "lradj": "type1",
        "factor": 1,
        "freq": "h",
    },
    "Electricity": {
        "data": "custom",
        "enc_in": 321,
        "e_layers": 3,
        "d_model": 512,
        "d_ff": 512,
        "n_heads": 8,
        "dropout": 0.1,
        "batch_size": 16,
        "learning_rate": 5e-4,
        "train_epochs": 10,
        "patience": 3,
        "lradj": "type1",
        "factor": 1,
        "freq": "h",
    },
}


def itransformer_config(
    name,
    horizon,
):
    r = RECIPES[
        name
    ]

    return SimpleNamespace(
        task_name=
            "long_term_forecast",
        seq_len=
            DIRECT_SEQ_LEN,
        label_len=
            LABEL_LEN,
        pred_len=
            int(
                horizon
            ),
        output_attention=
            False,
        enc_in=
            r[
                "enc_in"
            ],
        dec_in=
            r[
                "enc_in"
            ],
        c_out=
            r[
                "enc_in"
            ],
        d_model=
            r[
                "d_model"
            ],
        embed=
            "timeF",
        freq=
            r[
                "freq"
            ],
        dropout=
            r[
                "dropout"
            ],
        factor=
            r[
                "factor"
            ],
        n_heads=
            r[
                "n_heads"
            ],
        d_ff=
            r[
                "d_ff"
            ],
        activation=
            "gelu",
        e_layers=
            r[
                "e_layers"
            ],
        d_layers=
            1,
        class_strategy=
            "projection",
        use_norm=
            1,
    )


def build_itransformer(
    name,
    horizon,
):
    return OfficialITransformer(
        itransformer_config(
            name,
            horizon,
        )
    ).float().to(
        DEVICE
    )


def exp21_ckpt_path(
    name,
    horizon,
):
    return (
        EXP21_ROOT
        / "checkpoints"
        / (
            f"{name}_H{horizon}_"
            f"seed{DIRECT_SEED}.pt"
        )
    )


def exp21_reference(
    name,
    horizon,
):
    hit = EXP21_SUMMARY[
        (
            EXP21_SUMMARY[
                "Dataset"
            ]
            == name
        )
        & (
            EXP21_SUMMARY[
                "Horizon"
            ]
            == horizon
        )
    ]

    if len(
        hit
    ) != 1:
        raise RuntimeError(
            f"Expected one Experiment 21 row "
            f"for {name} H={horizon}."
        )

    return hit.iloc[
        0
    ]


,Dataset,Horizon,BestEpoch,Test_MSE,Test_MAE
4,ETTh1,96,1,0.392304,0.407334
5,ETTh1,192,1,0.442639,0.434737
6,ETTh1,336,1,0.489263,0.461457
7,ETTh1,720,1,0.506507,0.492894
0,ETTm1,96,1,0.336712,0.373422
1,ETTm1,192,3,0.394380,0.402333
2,ETTm1,336,2,0.424327,0.422129
3,ETTm1,720,3,0.496095,0.461646
12,Electricity,96,7,0.148275,0.239707
13,Electricity,192,7,0.165100,0.256106


## 3. Dataset paths and exact full-split normalization

In [4]:

DATA_PATH_CANDIDATES = {
    "ETTm1": [
        Path("/data/dataset/ETTm1.csv"),
        Path(
            "/data/Time-Series-Library/"
            "dataset/ETT-small/ETTm1.csv"
        ),
        Path(
            "/data/Time-Series-Library_v2/"
            "dataset/ETT-small/ETTm1.csv"
        ),
    ],
    "ETTh1": [
        Path("/data/dataset/ETTh1.csv"),
        Path(
            "/data/Time-Series-Library/"
            "dataset/ETT-small/ETTh1.csv"
        ),
        Path(
            "/data/Time-Series-Library_v2/"
            "dataset/ETT-small/ETTh1.csv"
        ),
    ],
    "Weather": [
        Path("/data/dataset/weather.csv"),
        Path("/data/dataset/weather/weather.csv"),
        Path(
            "/data/Time-Series-Library/"
            "dataset/weather/weather.csv"
        ),
        Path(
            "/data/Time-Series-Library_v2/"
            "dataset/weather/weather.csv"
        ),
    ],
    "Electricity": [
        Path("/data/dataset/electricity.csv"),
        Path(
            "/data/dataset/electricity/"
            "electricity.csv"
        ),
        Path(
            "/data/Time-Series-Library/"
            "dataset/electricity/electricity.csv"
        ),
        Path(
            "/data/Time-Series-Library_v2/"
            "dataset/electricity/electricity.csv"
        ),
    ],
}

DATA_PATHS = {}

for name, candidates in (
    DATA_PATH_CANDIDATES.items()
):
    hit = next(
        (
            p
            for p in candidates
            if p.is_file()
        ),
        None,
    )

    DATA_PATHS[
        name
    ] = hit

    print(
        f"{name:11s}:",
        hit if hit is not None else "NOT FOUND",
    )

if any(
    p is None
    for p in DATA_PATHS.values()
):
    raise FileNotFoundError(
        "At least one required dataset CSV is missing."
    )


def standard_boundaries(
    name,
    n,
):
    if name == "ETTh1":
        train_end = (
            12
            * 30
            * 24
        )

        val_end = (
            (
                12
                + 4
            )
            * 30
            * 24
        )

        test_end = (
            (
                12
                + 8
            )
            * 30
            * 24
        )

    elif name == "ETTm1":
        unit = (
            30
            * 24
            * 4
        )

        train_end = (
            12
            * unit
        )

        val_end = (
            (
                12
                + 4
            )
            * unit
        )

        test_end = (
            (
                12
                + 8
            )
            * unit
        )

    else:
        train_end = int(
            n
            * 0.7
        )

        num_test = int(
            n
            * 0.2
        )

        num_val = (
            n
            - train_end
            - num_test
        )

        val_end = (
            train_end
            + num_val
        )

        test_end = n

    if test_end > n:
        raise ValueError(
            f"{name}: dataset is shorter than "
            f"the official split. "
            f"Need {test_end}, found {n}."
        )

    return (
        train_end,
        val_end,
        test_end,
    )


def prepare_data(
    name,
):
    path = DATA_PATHS[
        name
    ]

    df = pd.read_csv(
        path
    )

    if "date" not in df.columns:
        raise ValueError(
            f"{name}: no date column."
        )

    if (
        name
        in [
            "Weather",
            "Electricity",
        ]
    ):
        if "OT" not in df.columns:
            raise ValueError(
                f"{name}: no OT column."
            )

        cols = list(
            df.columns
        )

        cols.remove(
            "OT"
        )

        cols.remove(
            "date"
        )

        df = df[
            ["date"]
            + cols
            + ["OT"]
        ].copy()

    value_cols = list(
        df.columns[
            1:
        ]
    )

    raw = df[
        value_cols
    ].to_numpy(
        dtype=np.float64
    )

    (
        train_end,
        val_end,
        test_end,
    ) = standard_boundaries(
        name,
        len(
            raw
        ),
    )

    scaler = StandardScaler()

    scaler.fit(
        raw[
            :train_end
        ]
    )

    z = scaler.transform(
        raw
    ).astype(
        np.float32
    )

    dates = pd.to_datetime(
        df[
            "date"
        ].values
    )

    marks = official_time_features(
        dates,
        freq=
            RECIPES[
                name
            ][
                "freq"
            ],
    ).transpose(
        1,
        0,
    ).astype(
        np.float32
    )

    C = raw.shape[
        1
    ]

    if (
        C
        != RECIPES[
            name
        ][
            "enc_in"
        ]
    ):
        raise ValueError(
            f"{name}: expected "
            f"{RECIPES[name]['enc_in']} channels, "
            f"found {C}."
        )

    return {
        "name":
            name,
        "path":
            path,
        "columns":
            value_cols,
        "raw":
            raw,
        "z":
            z,
        "marks":
            marks,
        "n_channels":
            C,
        "train_end":
            train_end,
        "val_end":
            val_end,
        "test_end":
            test_end,
        "full_scaler":
            scaler,
    }


DATA = {
    name:
        prepare_data(
            name
        )
    for name in DATASETS
}

display(
    pd.DataFrame([
        {
            "Dataset":
                name,
            "RowsUsed":
                d[
                    "test_end"
                ],
            "Channels":
                d[
                    "n_channels"
                ],
            "TrainEnd":
                d[
                    "train_end"
                ],
            "ValEnd":
                d[
                    "val_end"
                ],
            "TestEnd":
                d[
                    "test_end"
                ],
            "TimeFeatures":
                d[
                    "marks"
                ].shape[
                    1
                ],
        }
        for name, d
        in DATA.items()
    ])
)


ETTm1      : /data/dataset/ETTm1.csv
ETTh1      : /data/Time-Series-Library/dataset/ETT-small/ETTh1.csv
Weather    : /data/Time-Series-Library/dataset/weather/weather.csv
Electricity: /data/Time-Series-Library/dataset/electricity/electricity.csv


,Dataset,RowsUsed,Channels,TrainEnd,ValEnd,TestEnd,TimeFeatures
0,ETTm1,57600,7,34560,46080,57600,5
1,ETTh1,14400,7,8640,11520,14400,4
2,Weather,52696,21,36887,42157,52696,4
3,Electricity,26304,321,18412,21044,26304,4


## 4. Prefix-only normalization for chronological OOF folds

In [5]:

def prefix_normalize(
    raw,
    prefix,
):
    scaler = StandardScaler()

    scaler.fit(
        raw[
            :prefix
        ]
    )

    z = scaler.transform(
        raw
    ).astype(
        np.float32
    )

    return (
        z,
        scaler,
    )


## 5. iTransformer direct utilities

In [6]:

def set_seed(
    seed,
):
    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            seed
        )

    torch.backends.cudnn.benchmark = False


def load_torch(
    path,
):
    try:
        return torch.load(
            path,
            map_location=DEVICE,
            weights_only=False,
        )
    except TypeError:
        return torch.load(
            path,
            map_location=DEVICE,
        )


def load_exp21_direct(
    name,
    horizon,
):
    path = exp21_ckpt_path(
        name,
        horizon,
    )

    ckpt = load_torch(
        path
    )

    model = build_itransformer(
        name,
        horizon,
    )

    model.load_state_dict(
        ckpt[
            "StateDict"
        ]
    )

    model.eval()

    return (
        model,
        ckpt,
    )


def eval_anchors(
    start,
    end,
    horizon,
    stride=1,
):
    return np.arange(
        max(
            int(
                start
            ),
            DIRECT_SEQ_LEN,
        ),
        int(
            end
        )
        - int(
            horizon
        )
        + 1,
        int(
            stride
        ),
        dtype=np.int64,
    )


def make_direct_batch(
    z,
    marks,
    anchors,
    horizon,
):
    anchors = np.asarray(
        anchors,
        dtype=np.int64,
    )

    x_idx = (
        anchors[
            :,
            None
        ]
        - DIRECT_SEQ_LEN
        + np.arange(
            DIRECT_SEQ_LEN
        )[
            None,
            :
        ]
    )

    y_idx = (
        anchors[
            :,
            None
        ]
        - LABEL_LEN
        + np.arange(
            LABEL_LEN
            + horizon
        )[
            None,
            :
        ]
    )

    x = z[
        x_idx,
        :
    ].astype(
        np.float32
    )

    y = z[
        y_idx,
        :
    ].astype(
        np.float32
    )

    x_mark = marks[
        x_idx,
        :
    ].astype(
        np.float32
    )

    y_mark = marks[
        y_idx,
        :
    ].astype(
        np.float32
    )

    return (
        x,
        y,
        x_mark,
        y_mark,
    )


def forward_direct(
    model,
    x,
    y,
    x_mark,
    y_mark,
    horizon,
):
    x_t = torch.from_numpy(
        x
    ).to(
        DEVICE
    )

    y_t = torch.from_numpy(
        y
    ).to(
        DEVICE
    )

    xm_t = torch.from_numpy(
        x_mark
    ).to(
        DEVICE
    )

    ym_t = torch.from_numpy(
        y_mark
    ).to(
        DEVICE
    )

    dec_inp = torch.zeros_like(
        y_t[
            :,
            -horizon:,
            :
        ]
    )

    dec_inp = torch.cat(
        [
            y_t[
                :,
                :LABEL_LEN,
                :
            ],
            dec_inp,
        ],
        dim=1,
    )

    pred = model(
        x_t,
        xm_t,
        dec_inp,
        ym_t,
    )

    pred = pred[
        :,
        -horizon:,
        :
    ].float()

    true = y_t[
        :,
        -horizon:,
        :
    ].float()

    return (
        pred,
        true,
        x_t,
        y_t,
        xm_t,
        ym_t,
        dec_inp,
    )


@torch.no_grad()
def direct_residual_block(
    model,
    z,
    marks,
    anchors,
    horizon,
):
    (
        x,
        y,
        x_mark,
        y_mark,
    ) = make_direct_batch(
        z,
        marks,
        anchors,
        horizon,
    )

    (
        pred,
        true,
        x_t,
        y_t,
        xm_t,
        ym_t,
        dec_inp,
    ) = forward_direct(
        model,
        x,
        y,
        x_mark,
        y_mark,
        horizon,
    )

    current = x_t[
        :,
        -1:,
        :
    ].float()

    pred_residual = (
        pred
        - current
    )

    true_residual = (
        true
        - current
    )

    del (
        true,
        x_t,
        y_t,
        xm_t,
        ym_t,
        dec_inp,
    )

    return (
        pred_residual,
        true_residual,
    )


@torch.no_grad()
def evaluate_direct_parity(
    name,
    horizon,
    model,
):
    d = DATA[
        name
    ]

    anchors = eval_anchors(
        d[
            "val_end"
        ],
        d[
            "test_end"
        ],
        horizon,
        stride=1,
    )

    sse = 0.0
    sae = 0.0
    n = 0

    block = DIRECT_ANCHOR_BLOCK[
        name
    ]

    for i in range(
        0,
        len(
            anchors
        ),
        block,
    ):
        a = anchors[
            i:
            i+block
        ]

        pred_r, true_r = (
            direct_residual_block(
                model,
                d[
                    "z"
                ],
                d[
                    "marks"
                ],
                a,
                horizon,
            )
        )

        e = (
            pred_r
            - true_r
        )

        sse += float(
            (
                e
                * e
            ).sum()
        )

        sae += float(
            e.abs().sum()
        )

        n += e.numel()

        del (
            pred_r,
            true_r,
            e,
        )

    return (
        sse
        / n,
        sae
        / n,
        len(
            anchors
        ),
    )



## 6. Direct parity preflight

Experiment 22에서 사용하는 manual chronological window가
Experiment 21의 official data provider와 동일한 prediction set을 만드는지 먼저 확인합니다.

각 조건에서

$$
\left|
\mathrm{MSE}_{22,\mathrm{direct}}
-
\mathrm{MSE}_{21}
\right|
<
2\times10^{-5}
$$

를 요구합니다.

이 검사를 통과하기 전에는 expensive cross-fitting을 시작하지 않습니다.


In [7]:

PARITY_PATH = (
    DIRS[
        "artifacts"
    ]
    / "direct_parity.csv"
)

parity_rows = []

for name, horizon in TASKS:
    model, ckpt = load_exp21_direct(
        name,
        horizon,
    )

    mse, mae, windows = (
        evaluate_direct_parity(
            name,
            horizon,
            model,
        )
    )

    ref = exp21_reference(
        name,
        horizon,
    )

    mse_diff = abs(
        mse
        - float(
            ref[
                "Test_MSE"
            ]
        )
    )

    mae_diff = abs(
        mae
        - float(
            ref[
                "Test_MAE"
            ]
        )
    )

    parity_rows.append({
        "Dataset":
            name,
        "Horizon":
            horizon,
        "Exp21_MSE":
            float(
                ref[
                    "Test_MSE"
                ]
            ),
        "Manual_MSE":
            mse,
        "MSEAbsDiff":
            mse_diff,
        "Exp21_MAE":
            float(
                ref[
                    "Test_MAE"
                ]
            ),
        "Manual_MAE":
            mae,
        "MAEAbsDiff":
            mae_diff,
        "Windows":
            windows,
        "Pass":
            mse_diff
            < DIRECT_PARITY_TOL,
    })

    print(
        f"{name:11s} H={horizon:3d} | "
        f"Exp21={float(ref['Test_MSE']):.8f} | "
        f"manual={mse:.8f} | "
        f"diff={mse_diff:.2e} | "
        f"{'PASS' if mse_diff < DIRECT_PARITY_TOL else 'FAIL'}"
    )

    del (
        model,
        ckpt,
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

parity_df = pd.DataFrame(
    parity_rows
)

parity_df.to_csv(
    PARITY_PATH,
    index=False,
)

display(
    parity_df
)

failed = parity_df[
    ~parity_df[
        "Pass"
    ]
]

if len(
    failed
):
    raise RuntimeError(
        "Direct parity failed. "
        "Do not start cross-fitting until this is fixed."
    )

print(
    "PASS: all 16 direct baselines match Experiment 21."
)


ETTm1       H= 96 | Exp21=0.33671204 | manual=0.33671205 | diff=7.84e-09 | PASS
ETTm1       H=192 | Exp21=0.39437993 | manual=0.39437989 | diff=4.29e-08 | PASS
ETTm1       H=336 | Exp21=0.42432655 | manual=0.42432651 | diff=3.81e-08 | PASS
ETTm1       H=720 | Exp21=0.49609469 | manual=0.49609468 | diff=1.20e-08 | PASS
ETTh1       H= 96 | Exp21=0.39230357 | manual=0.39230347 | diff=1.04e-07 | PASS
ETTh1       H=192 | Exp21=0.44263935 | manual=0.44263931 | diff=3.33e-08 | PASS
ETTh1       H=336 | Exp21=0.48926283 | manual=0.48926283 | diff=5.58e-09 | PASS
ETTh1       H=720 | Exp21=0.50650688 | manual=0.50650684 | diff=4.11e-08 | PASS
Weather     H= 96 | Exp21=0.17328721 | manual=0.17328719 | diff=2.38e-08 | PASS
Weather     H=192 | Exp21=0.22443607 | manual=0.22443608 | diff=1.06e-08 | PASS
Weather     H=336 | Exp21=0.28273247 | manual=0.28273248 | diff=8.18e-09 | PASS
Weather     H=720 | Exp21=0.35790776 | manual=0.35790778 | diff=1.75e-08 | PASS
Electricity H= 96 | Exp21=0.14827520 | m

,Dataset,Horizon,Exp21_MSE,Manual_MSE,MSEAbsDiff,Exp21_MAE,Manual_MAE,MAEAbsDiff,Windows,Pass
0,ETTm1,96,0.336712,0.336712,7.835989e-09,0.373422,0.373422,1.574651e-08,11425,True
1,ETTm1,192,0.394380,0.394380,4.289268e-08,0.402333,0.402333,1.446892e-08,11329,True
2,ETTm1,336,0.424327,0.424327,3.805196e-08,0.422129,0.422129,1.184643e-08,11185,True
3,ETTm1,720,0.496095,0.496095,1.202157e-08,0.461646,0.461646,1.382447e-08,10801,True
4,ETTh1,96,0.392304,0.392303,1.038345e-07,0.407334,0.407334,5.240847e-08,2785,True
5,ETTh1,192,0.442639,0.442639,3.329982e-08,0.434737,0.434737,1.221880e-08,2689,True
6,ETTh1,336,0.489263,0.489263,5.577535e-09,0.461457,0.461457,5.536748e-09,2545,True
7,ETTh1,720,0.506507,0.506507,4.108821e-08,0.492894,0.492894,4.236591e-09,2161,True
8,Weather,96,0.173287,0.173287,2.378435e-08,0.212210,0.212210,4.415642e-09,10444,True
9,Weather,192,0.224436,0.224436,1.061105e-08,0.257719,0.257719,7.659541e-09,10348,True


PASS: all 16 direct baselines match Experiment 21.


## 7. Frozen predictive retriever

In [8]:

def inv_softplus(
    x,
):
    return math.log(
        math.exp(
            float(
                x
            )
        )
        - 1.0
    )


class PredictivePatchEncoder(
    nn.Module
):
    def __init__(
        self,
    ):
        super().__init__()

        self.patch_proj = nn.Linear(
            REP_PATCH_LEN,
            REP_D_MODEL,
        )

        self.pos_embed = nn.Parameter(
            torch.zeros(
                1,
                REP_NUM_PATCHES,
                REP_D_MODEL,
            )
        )

        nn.init.trunc_normal_(
            self.pos_embed,
            std=0.02,
        )

        layer = nn.TransformerEncoderLayer(
            d_model=
                REP_D_MODEL,
            nhead=
                REP_N_HEADS,
            dim_feedforward=
                REP_D_FF,
            dropout=
                REP_DROPOUT,
            activation=
                "gelu",
            batch_first=
                True,
            norm_first=
                True,
        )

        self.encoder = nn.TransformerEncoder(
            layer,
            num_layers=
                REP_LAYERS,
        )

        self.norm = nn.LayerNorm(
            REP_D_MODEL
        )

        self.proj = nn.Linear(
            REP_D_MODEL,
            REP_DIM,
        )

    def forward(
        self,
        x,
    ):
        p = x.unfold(
            1,
            REP_PATCH_LEN,
            REP_PATCH_STRIDE,
        )

        h = (
            self.patch_proj(
                p
            )
            + self.pos_embed[
                :,
                :p.shape[
                    1
                ],
            ]
        )

        h = self.encoder(
            h
        ).mean(
            dim=1
        )

        h = self.proj(
            self.norm(
                h
            )
        )

        return F.normalize(
            h,
            dim=-1,
            eps=1e-8,
        )


class EmbeddingOnlyRetriever(
    nn.Module
):
    def __init__(
        self,
    ):
        super().__init__()

        self.encoder = (
            PredictivePatchEncoder()
        )

        self.raw_gamma = nn.Parameter(
            torch.tensor(
                inv_softplus(
                    1.0
                ),
                dtype=torch.float32,
            )
        )

    @property
    def gamma(
        self,
    ):
        return F.softplus(
            self.raw_gamma
        )

    def encode(
        self,
        x,
    ):
        return self.encoder(
            x
        )


def ret_amp():
    if RETRIEVER_USE_AMP:
        return torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        )

    return nullcontext()


def full_retriever_ckpt_path(
    name,
    horizon,
):
    return (
        FROZEN_RET_ROOT
        / "full_retriever"
        / (
            f"{name}_H{horizon}_seed0.pt"
        )
    )


def fold_retriever_ckpt_path(
    name,
    horizon,
    fold,
):
    return (
        FROZEN_RET_ROOT
        / "fold_retriever"
        / (
            f"{name}_H{horizon}_"
            f"F{fold}_seed0.pt"
        )
    )


def load_frozen_retriever(
    path,
):
    ckpt = load_torch(
        path
    )

    model = EmbeddingOnlyRetriever().to(
        DEVICE
    )

    model.load_state_dict(
        ckpt[
            "StateDict"
        ]
    )

    model.eval()

    return (
        model,
        ckpt,
    )


## 8. Full prerequisite checkpoint audit

In [9]:

preflight_rows = []

for name, horizon in TASKS:
    preflight_rows.append({
        "Dataset":
            name,
        "Horizon":
            horizon,
        "Kind":
            "Exp21 direct",
        "Path":
            exp21_ckpt_path(
                name,
                horizon,
            ),
    })

    preflight_rows.append({
        "Dataset":
            name,
        "Horizon":
            horizon,
        "Kind":
            "Full retriever",
        "Path":
            full_retriever_ckpt_path(
                name,
                horizon,
            ),
    })

    for fold in range(
        1,
        4,
    ):
        preflight_rows.append({
            "Dataset":
                name,
            "Horizon":
                horizon,
            "Kind":
                f"Fold retriever F{fold}",
            "Path":
                fold_retriever_ckpt_path(
                    name,
                    horizon,
                    fold,
                ),
        })

preflight = pd.DataFrame(
    preflight_rows
)

preflight[
    "Exists"
] = preflight[
    "Path"
].map(
    lambda p:
        Path(
            p
        ).is_file()
)

display(
    preflight
)

missing = preflight[
    ~preflight[
        "Exists"
    ]
]

if len(
    missing
):
    for p in missing[
        "Path"
    ]:
        print(
            "MISSING:",
            p,
        )

    raise FileNotFoundError(
        "Required frozen checkpoint is missing."
    )

print(
    "PASS: all 80 prerequisite checkpoint references exist."
)


,Dataset,Horizon,Kind,Path,Exists
0,ETTm1,96,Exp21 direct,/data/dataset/strong_forecaster/itransformer_o...,True
1,ETTm1,96,Full retriever,/data/dataset/strong_forecaster/multidataset_c...,True
2,ETTm1,96,Fold retriever F1,/data/dataset/strong_forecaster/multidataset_c...,True
3,ETTm1,96,Fold retriever F2,/data/dataset/strong_forecaster/multidataset_c...,True
4,ETTm1,96,Fold retriever F3,/data/dataset/strong_forecaster/multidataset_c...,True
...,...,...,...,...,...
75,Electricity,720,Exp21 direct,/data/dataset/strong_forecaster/itransformer_o...,True
76,Electricity,720,Full retriever,/data/dataset/strong_forecaster/multidataset_c...,True
77,Electricity,720,Fold retriever F1,/data/dataset/strong_forecaster/multidataset_c...,True
78,Electricity,720,Fold retriever F2,/data/dataset/strong_forecaster/multidataset_c...,True


PASS: all 80 prerequisite checkpoint references exist.


## 9. Retrieval time-series utilities

In [10]:

def retrieval_anchor_batch(
    name,
):
    C = DATA[
        name
    ][
        "n_channels"
    ]

    return max(
        1,
        TARGET_RETRIEVAL_PAIRS
        // C,
    )


def batch_pattern(
    x,
):
    x = np.asarray(
        x,
        np.float32,
    )

    xc = (
        x
        - x.mean(
            axis=-1,
            keepdims=True,
        )
    )

    n = np.linalg.norm(
        xc,
        axis=-1,
        keepdims=True,
    )

    return np.where(
        n > EPS,
        xc
        / np.maximum(
            n,
            EPS,
        ),
        0.0,
    ).astype(
        np.float32
    )


def context7(
    x,
):
    x = np.asarray(
        x,
        np.float32,
    )

    short = max(
        8,
        RET_SEQ_LEN
        // 4,
    )

    m = x.mean(
        axis=-1
    )

    s = (
        x.std(
            axis=-1
        )
        + EPS
    )

    f1 = (
        x[
            ...,
            -1
        ]
        - m
    ) / s

    f2 = (
        x[
            ...,
            -short:
        ].mean(
            axis=-1
        )
        - m
    ) / s

    f3 = (
        x[
            ...,
            -1
        ]
        - x[
            ...,
            -short
        ]
    ) / s

    f4 = (
        x[
            ...,
            -1
        ]
        - x[
            ...,
            0
        ]
    ) / s

    df = np.diff(
        x,
        axis=-1,
    )

    ds = np.diff(
        x[
            ...,
            -short:
        ],
        axis=-1,
    )

    f5 = (
        ds.std(
            axis=-1
        )
        + EPS
    ) / (
        df.std(
            axis=-1
        )
        + EPS
    )

    t = np.linspace(
        -1.0,
        1.0,
        RET_SEQ_LEN,
        dtype=np.float32,
    )

    t = (
        t
        - t.mean()
    )

    f6 = (
        np.sum(
            t
            * (
                x
                - m[
                    ...,
                    None
                ]
            ),
            axis=-1,
        )
        / (
            np.sum(
                t
                * t
            )
            + EPS
        )
    ) / s

    a = x[
        ...,
        :-1
    ]

    b = x[
        ...,
        1:
    ]

    a = (
        a
        - a.mean(
            axis=-1,
            keepdims=True,
        )
    )

    b = (
        b
        - b.mean(
            axis=-1,
            keepdims=True,
        )
    )

    f7 = np.sum(
        a
        * b,
        axis=-1,
    ) / (
        np.sqrt(
            np.sum(
                a
                * a,
                axis=-1,
            )
            * np.sum(
                b
                * b,
                axis=-1,
            )
        )
        + EPS
    )

    return np.stack(
        [
            f1,
            f2,
            f3,
            f4,
            f5,
            f6,
            f7,
        ],
        axis=-1,
    ).astype(
        np.float32
    )


def extract_channel(
    z,
    c,
    anchors,
    horizon,
):
    anchors = np.asarray(
        anchors,
        np.int64,
    )

    pi = (
        anchors[
            :,
            None
        ]
        - RET_SEQ_LEN
        + np.arange(
            RET_SEQ_LEN
        )[
            None,
            :
        ]
    )

    fi = (
        anchors[
            :,
            None
        ]
        + np.arange(
            horizon
        )[
            None,
            :
        ]
    )

    past = z[
        pi,
        c,
    ].astype(
        np.float32
    )

    future = z[
        fi,
        c,
    ].astype(
        np.float32
    )

    current = z[
        anchors
        - 1,
        c,
    ].astype(
        np.float32
    )

    future_residual = (
        future
        - current[
            :,
            None
        ]
    ).astype(
        np.float32
    )

    return (
        past,
        future_residual,
    )


def build_memory(
    z,
    channels,
    boundary,
    horizon,
):
    memory_anchors = np.arange(
        RET_SEQ_LEN,
        int(
            boundary
        )
        - horizon
        + 1,
        MEMORY_STRIDE,
        dtype=np.int64,
    )

    if len(
        memory_anchors
    ) < TOP_K:
        raise ValueError(
            "Insufficient admissible memory."
        )

    past = np.empty(
        (
            channels,
            len(
                memory_anchors
            ),
            RET_SEQ_LEN,
        ),
        dtype=np.float32,
    )

    pattern = np.empty_like(
        past
    )

    future = np.empty(
        (
            channels,
            len(
                memory_anchors
            ),
            horizon,
        ),
        dtype=np.float32,
    )

    for c in range(
        channels
    ):
        p, f = extract_channel(
            z,
            c,
            memory_anchors,
            horizon,
        )

        past[
            c
        ] = p

        pattern[
            c
        ] = batch_pattern(
            p
        )

        future[
            c
        ] = f

    return {
        "anchors":
            memory_anchors,
        "past":
            past,
        "pattern":
            pattern,
        "future":
            future,
        "M":
            len(
                memory_anchors
            ),
        "boundary":
            int(
                boundary
            ),
    }


def query_pairs(
    z,
    anchors,
    channels,
    horizon,
):
    anchors = np.asarray(
        anchors,
        np.int64,
    )

    channels = np.asarray(
        channels,
        np.int64,
    )

    pi = (
        anchors[
            :,
            None
        ]
        - RET_SEQ_LEN
        + np.arange(
            RET_SEQ_LEN
        )[
            None,
            :
        ]
    )

    fi = (
        anchors[
            :,
            None
        ]
        + np.arange(
            horizon
        )[
            None,
            :
        ]
    )

    past = z[
        pi,
        channels[
            :,
            None
        ],
    ].astype(
        np.float32
    )

    future = z[
        fi,
        channels[
            :,
            None
        ],
    ].astype(
        np.float32
    )

    current = z[
        anchors
        - 1,
        channels,
    ].astype(
        np.float32
    )

    true_residual = (
        future
        - current[
            :,
            None
        ]
    ).astype(
        np.float32
    )

    return (
        past,
        batch_pattern(
            past
        ),
        context7(
            past
        ),
        true_residual,
    )


## 10. Cached memory embeddings

In [11]:

@torch.no_grad()
def encode_np(
    model,
    x,
    chunk=512,
):
    parts = []

    for i in range(
        0,
        len(
            x
        ),
        chunk,
    ):
        t = torch.from_numpy(
            x[
                i:
                i+chunk
            ]
        ).to(
            DEVICE
        )

        with ret_amp():
            e = model.encode(
                t
            ).float()

        parts.append(
            e.cpu()
        )

        del (
            t,
            e,
        )

    return torch.cat(
        parts,
        dim=0,
    ).numpy().astype(
        np.float32
    )


def memory_embedding_path(
    name,
    horizon,
    tag,
):
    return (
        DIRS[
            "memory_emb"
        ]
        / (
            f"{name}_H{horizon}_"
            f"{tag}_emb.npy"
        )
    )


@torch.no_grad()
def memory_gpu_cached(
    name,
    horizon,
    tag,
    model,
    memory,
    channels,
):
    path = memory_embedding_path(
        name,
        horizon,
        tag,
    )

    expected = (
        channels,
        memory[
            "M"
        ],
        REP_DIM,
    )

    emb_np = None

    if (
        path.exists()
        and RESUME
        and not FORCE
    ):
        candidate = np.load(
            path,
            mmap_mode=None,
        )

        if (
            tuple(
                candidate.shape
            )
            == expected
        ):
            emb_np = candidate.astype(
                np.float32,
                copy=False,
            )

            print(
                "Loaded memory embedding:",
                path.name,
            )

    if emb_np is None:
        emb_np = np.empty(
            expected,
            dtype=np.float32,
        )

        print(
            "Building memory embedding:",
            path.name,
            expected,
        )

        for c in range(
            channels
        ):
            emb_np[
                c
            ] = encode_np(
                model,
                memory[
                    "past"
                ][
                    c
                ],
            )

            if (
                c == 0
                or (
                    c + 1
                )
                % 50
                == 0
                or (
                    c + 1
                    == channels
                )
            ):
                print(
                    f"  channel "
                    f"{c+1}/{channels}"
                )

        np.save(
            path,
            emb_np,
        )

    return {
        "emb":
            torch.from_numpy(
                emb_np
            ).to(
                DEVICE
            ),
        "pattern":
            torch.from_numpy(
                memory[
                    "pattern"
                ]
            ).to(
                DEVICE
            ),
        "future":
            torch.from_numpy(
                memory[
                    "future"
                ]
            ).to(
                DEVICE
            ),
    }


## 11. Frozen retrieval inference

In [12]:

@torch.no_grad()
def retrieve(
    model,
    memory_gpu_obj,
    z,
    anchors,
    channels,
    horizon,
):
    (
        past,
        pattern,
        ctx,
        true,
    ) = query_pairs(
        z,
        anchors,
        channels,
        horizon,
    )

    past_t = torch.from_numpy(
        past
    ).to(
        DEVICE
    )

    pattern_t = torch.from_numpy(
        pattern
    ).to(
        DEVICE
    )

    with ret_amp():
        qemb = model.encode(
            past_t
        )

    qemb = qemb.float()

    memb = memory_gpu_obj[
        "emb"
    ][
        channels
    ]

    sim = torch.bmm(
        qemb[
            :,
            None,
            :
        ],
        memb.transpose(
            1,
            2,
        ),
    ).squeeze(
        1
    )

    score = (
        model.gamma
        * sim
    )

    idx = torch.topk(
        score,
        TOP_K,
        dim=1,
    ).indices

    row = torch.arange(
        len(
            channels
        ),
        device=DEVICE,
    )[
        :,
        None
    ]

    pfull = torch.bmm(
        pattern_t[
            :,
            None,
            :
        ],
        memory_gpu_obj[
            "pattern"
        ][
            channels
        ].transpose(
            1,
            2,
        ),
    ).squeeze(
        1
    )

    return {
        "score":
            score[
                row,
                idx
            ],
        "sim":
            sim[
                row,
                idx
            ],
        "pattern":
            pfull[
                row,
                idx
            ],
        "cand":
            memory_gpu_obj[
                "future"
            ][
                channels[
                    :,
                    None
                ],
                idx,
            ],
        "ctx":
            torch.from_numpy(
                ctx
            ).to(
                DEVICE
            ),
        "true":
            torch.from_numpy(
                true
            ).to(
                DEVICE
            ),
    }


## 12. Frozen 26-dimensional adaptive gate

In [13]:

class CrossFitAdaptiveGate(
    nn.Module
):
    def __init__(
        self,
    ):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(
                GATE_DIM,
                64,
            ),
            nn.LayerNorm(
                64
            ),
            nn.GELU(),
            nn.Dropout(
                0.1
            ),
            nn.Linear(
                64,
                32,
            ),
            nn.GELU(),
            nn.Dropout(
                0.1
            ),
            nn.Linear(
                32,
                1,
            ),
        )

        nn.init.normal_(
            self.net[
                -1
            ].weight,
            mean=0.0,
            std=1e-3,
        )

        nn.init.constant_(
            self.net[
                -1
            ].bias,
            math.log(
                0.1
                / 0.9
            ),
        )

    def forward(
        self,
        x,
    ):
        return torch.sigmoid(
            self.net(
                x
            ).squeeze(
                -1
            )
        )


def score_entropy(
    s,
):
    p = torch.softmax(
        s,
        dim=1,
    )

    return (
        -(
            p
            * torch.log(
                p.clamp_min(
                    1e-8
                )
            )
        ).sum(
            dim=1
        )
        / math.log(
            TOP_K
        )
    )


def feature_cosine(
    a,
    b,
):
    return (
        (
            a
            * b
        ).sum(
            dim=1
        )
        / (
            torch.sqrt(
                (
                    a
                    * a
                ).sum(
                    dim=1
                )
                + 1e-8
            )
            * torch.sqrt(
                (
                    b
                    * b
                ).sum(
                    dim=1
                )
                + 1e-8
            )
        )
    )


def gate_features(
    r,
    retrieval,
    direct,
):
    s = r[
        "score"
    ]

    sim = r[
        "sim"
    ]

    pattern = r[
        "pattern"
    ]

    sorted_s = torch.sort(
        s,
        dim=1,
        descending=True,
    ).values

    cand_std = r[
        "cand"
    ].std(
        dim=1,
        unbiased=False,
    )

    disp_rms = torch.sqrt(
        (
            cand_std
            * cand_std
        ).mean(
            dim=1
        )
        + 1e-8
    )

    disp_mean = cand_std.mean(
        dim=1
    )

    direct_rms = torch.sqrt(
        (
            direct
            * direct
        ).mean(
            dim=1
        )
        + 1e-8
    )

    retrieval_rms = torch.sqrt(
        (
            retrieval
            * retrieval
        ).mean(
            dim=1
        )
        + 1e-8
    )

    disagreement = (
        retrieval
        - direct
    )

    disagreement_rms = torch.sqrt(
        (
            disagreement
            * disagreement
        ).mean(
            dim=1
        )
        + 1e-8
    )

    relative_disagreement = (
        disagreement_rms
        / (
            direct_rms
            + retrieval_rms
            + 1e-6
        )
    )

    scalars = torch.stack(
        [
            s.mean(
                dim=1
            ),
            s.std(
                dim=1,
                unbiased=False,
            ),
            s.max(
                dim=1
            ).values,
            sorted_s[
                :,
                0
            ]
            - sorted_s[
                :,
                1
            ],
            s.max(
                dim=1
            ).values
            - s.mean(
                dim=1
            ),
            score_entropy(
                s
            ),
            sim.mean(
                dim=1
            ),
            sim.std(
                dim=1,
                unbiased=False,
            ),
            sim.max(
                dim=1
            ).values,
            pattern.mean(
                dim=1
            ),
            pattern.std(
                dim=1,
                unbiased=False,
            ),
            pattern.max(
                dim=1
            ).values,
            disp_rms,
            disp_mean,
            direct_rms,
            retrieval_rms,
            disagreement_rms,
            relative_disagreement,
            feature_cosine(
                direct,
                retrieval,
            ),
        ],
        dim=1,
    )

    out = torch.cat(
        [
            r[
                "ctx"
            ],
            scalars,
        ],
        dim=1,
    )

    if out.shape[
        1
    ] != GATE_DIM:
        raise RuntimeError(
            f"Gate feature dimension mismatch: "
            f"{out.shape}"
        )

    return out


def abc_terms(
    direct,
    retrieval,
    true,
):
    e = (
        direct
        - true
    )

    delta = (
        retrieval
        - direct
    )

    return torch.stack(
        [
            (
                e
                * e
            ).mean(
                dim=1
            ),
            (
                e
                * delta
            ).mean(
                dim=1
            ),
            (
                delta
                * delta
            ).mean(
                dim=1
            ),
        ],
        dim=1,
    )


## 13. Fold-specific iTransformer training

In [14]:

def fold_direct_path(
    name,
    horizon,
    fold,
):
    return (
        DIRS[
            "fold_direct"
        ]
        / (
            f"{name}_H{horizon}_"
            f"F{fold}_itransformer.pt"
        )
    )


def history_path(
    name,
    horizon,
    fold,
):
    return (
        DIRS[
            "history"
        ]
        / (
            f"{name}_H{horizon}_"
            f"F{fold}_direct_history.csv"
        )
    )


def train_anchors(
    prefix,
    horizon,
):
    return np.arange(
        DIRECT_SEQ_LEN,
        int(
            prefix
        )
        - horizon
        + 1,
        dtype=np.int64,
    )


def train_fold_itransformer(
    name,
    horizon,
    z,
    marks,
    prefix,
    fixed_epochs,
    fold,
):
    path = fold_direct_path(
        name,
        horizon,
        fold,
    )

    if (
        path.exists()
        and RESUME
        and not FORCE
    ):
        ckpt = load_torch(
            path
        )

        model = build_itransformer(
            name,
            horizon,
        )

        model.load_state_dict(
            ckpt[
                "StateDict"
            ]
        )

        model.eval()

        print(
            "Loaded fold direct:",
            path.name,
        )

        return (
            model,
            ckpt,
        )

    r = RECIPES[
        name
    ]

    seed = (
        CROSSFIT_SEED
        + horizon
        * 100
        + fold
        * 17
        + int(
            prefix
        )
        % 997
    )

    set_seed(
        seed
    )

    model = build_itransformer(
        name,
        horizon,
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=
            r[
                "learning_rate"
            ],
    )

    anchors = train_anchors(
        prefix,
        horizon,
    )

    rng = np.random.default_rng(
        seed
        + 1
    )

    history = []

    for epoch in range(
        1,
        int(
            fixed_epochs
        )
        + 1,
    ):
        model.train()

        order = rng.permutation(
            len(
                anchors
            )
        )

        usable = (
            len(
                order
            )
            // r[
                "batch_size"
            ]
        ) * r[
            "batch_size"
        ]

        order = order[
            :usable
        ]

        losses = []
        t0 = time.time()

        for left in range(
            0,
            usable,
            r[
                "batch_size"
            ],
        ):
            ids = order[
                left:
                left
                + r[
                    "batch_size"
                ]
            ]

            a = anchors[
                ids
            ]

            (
                x,
                y,
                x_mark,
                y_mark,
            ) = make_direct_batch(
                z,
                marks,
                a,
                horizon,
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            (
                pred,
                true,
                x_t,
                y_t,
                xm_t,
                ym_t,
                dec_inp,
            ) = forward_direct(
                model,
                x,
                y,
                x_mark,
                y_mark,
                horizon,
            )

            loss = F.mse_loss(
                pred,
                true,
            )

            loss.backward()
            optimizer.step()

            losses.append(
                float(
                    loss.item()
                )
            )

            del (
                pred,
                true,
                x_t,
                y_t,
                xm_t,
                ym_t,
                dec_inp,
                loss,
            )

        train_mse = float(
            np.mean(
                losses
            )
        )

        # Same type1 schedule as Experiment 21.
        args = SimpleNamespace(
            learning_rate=
                r[
                    "learning_rate"
                ],
            lradj=
                r[
                    "lradj"
                ],
            train_epochs=
                r[
                    "train_epochs"
                ],
        )

        official_adjust_lr(
            optimizer,
            epoch,
            args,
        )

        current_lr = optimizer.param_groups[
            0
        ][
            "lr"
        ]

        history.append({
            "Epoch":
                epoch,
            "TrainMSE":
                train_mse,
            "LR":
                current_lr,
            "Seconds":
                time.time()
                - t0,
        })

        pd.DataFrame(
            history
        ).to_csv(
            history_path(
                name,
                horizon,
                fold,
            ),
            index=False,
        )

        print(
            f"Fold direct {name:11s} "
            f"H={horizon:3d} F{fold} "
            f"ep={epoch:02d}/{fixed_epochs} "
            f"train={train_mse:.6f} "
            f"lr={current_lr:.3e}"
        )

    model.eval()

    state = {
        k:
            v.detach()
            .cpu()
            .clone()
        for k, v
        in model.state_dict().items()
    }

    ckpt = {
        "Dataset":
            name,
        "Horizon":
            horizon,
        "Fold":
            fold,
        "Prefix":
            int(
                prefix
            ),
        "FixedEpochs":
            int(
                fixed_epochs
            ),
        "Seed":
            seed,
        "StateDict":
            state,
    }

    torch.save(
        ckpt,
        path,
    )

    return (
        model,
        ckpt,
    )



## 14. Efficient gate-data collection

Direct backbone은 retrieval query batch보다 훨씬 큰 anchor block으로 한 번 계산합니다.

예를 들어 Electricity에서는:

- direct block: 16 anchors
- retrieval sub-batch: 약 2 anchors

따라서 같은 direct forward를 8번 반복하지 않습니다.

이 최적화는 prediction 값이나 training protocol을 바꾸지 않습니다.


In [15]:

@torch.no_grad()
def collect_gate_data(
    data,
    horizon,
    direct_model,
    retriever,
    memory_gpu_obj,
    z,
    anchors,
):
    name = data[
        "name"
    ]

    C = data[
        "n_channels"
    ]

    ret_block = retrieval_anchor_batch(
        name
    )

    direct_block = DIRECT_ANCHOR_BLOCK[
        name
    ]

    features = []
    abcs = []
    anchors_out = []
    channels_out = []

    for outer in range(
        0,
        len(
            anchors
        ),
        direct_block,
    ):
        a_big = anchors[
            outer:
            outer+direct_block
        ]

        direct_big, true_big = (
            direct_residual_block(
                direct_model,
                z,
                data[
                    "marks"
                ],
                a_big,
                horizon,
            )
        )

        # [A, H, C] -> [A, C, H]
        direct_big = direct_big.permute(
            0,
            2,
            1,
        ).contiguous()

        true_big = true_big.permute(
            0,
            2,
            1,
        ).contiguous()

        for inner in range(
            0,
            len(
                a_big
            ),
            ret_block,
        ):
            a = a_big[
                inner:
                inner+ret_block
            ]

            A = len(
                a
            )

            pair_anchor = np.repeat(
                a,
                C,
            )

            pair_channel = np.tile(
                np.arange(
                    C,
                    dtype=np.int64,
                ),
                A,
            )

            r = retrieve(
                retriever,
                memory_gpu_obj,
                z,
                pair_anchor,
                pair_channel,
                horizon,
            )

            retrieval = r[
                "cand"
            ].mean(
                dim=1
            )

            d = direct_big[
                inner:
                inner+A
            ].reshape(
                -1,
                horizon,
            )

            t = true_big[
                inner:
                inner+A
            ].reshape(
                -1,
                horizon,
            )

            # Strong consistency check:
            # direct true residual must match retrieval true residual.
            max_true_diff = float(
                (
                    t
                    - r[
                        "true"
                    ]
                ).abs().max()
            )

            if (
                max_true_diff
                > 2e-5
            ):
                raise RuntimeError(
                    f"True residual mismatch: "
                    f"{max_true_diff}"
                )

            feat = gate_features(
                r,
                retrieval,
                d,
            )

            abc = abc_terms(
                d,
                retrieval,
                t,
            )

            features.append(
                feat.cpu()
                .numpy()
                .astype(
                    np.float32
                )
            )

            abcs.append(
                abc.cpu()
                .numpy()
                .astype(
                    np.float32
                )
            )

            anchors_out.append(
                pair_anchor
            )

            channels_out.append(
                pair_channel
            )

            del (
                r,
                retrieval,
                d,
                t,
                feat,
                abc,
            )

        del (
            direct_big,
            true_big,
        )

    return {
        "feature":
            np.concatenate(
                features,
                axis=0,
            ),
        "abc":
            np.concatenate(
                abcs,
                axis=0,
            ),
        "anchor":
            np.concatenate(
                anchors_out,
                axis=0,
            ),
        "channel":
            np.concatenate(
                channels_out,
                axis=0,
            ),
    }


## 15. OOF and validation feature cache

In [16]:

def oof_cache_path(
    name,
    horizon,
    fold,
):
    return (
        DIRS[
            "oof"
        ]
        / (
            f"{name}_H{horizon}_"
            f"F{fold}_oof.npz"
        )
    )


def val_cache_path(
    name,
    horizon,
):
    return (
        DIRS[
            "validation"
        ]
        / (
            f"{name}_H{horizon}_"
            "validation.npz"
        )
    )


def build_oof_fold(
    data,
    horizon,
    fold,
    p0,
    p1,
    fixed_epochs,
):
    name = data[
        "name"
    ]

    C = data[
        "n_channels"
    ]

    out_path = oof_cache_path(
        name,
        horizon,
        fold,
    )

    if (
        out_path.exists()
        and RESUME
        and not FORCE
    ):
        obj = np.load(
            out_path
        )

        print(
            "Loaded OOF cache:",
            out_path.name,
        )

        return {
            key:
                obj[
                    key
                ]
            for key in [
                "feature",
                "abc",
                "anchor",
                "channel",
            ]
        }

    prefix = int(
        p0
        * data[
            "train_end"
        ]
    )

    oof_end = int(
        p1
        * data[
            "train_end"
        ]
    )

    z, prefix_scaler = prefix_normalize(
        data[
            "raw"
        ],
        prefix,
    )

    direct_model, _ = (
        train_fold_itransformer(
            name,
            horizon,
            z,
            data[
                "marks"
            ],
            prefix,
            fixed_epochs,
            fold,
        )
    )

    retriever, _ = load_frozen_retriever(
        fold_retriever_ckpt_path(
            name,
            horizon,
            fold,
        )
    )

    memory = build_memory(
        z,
        C,
        prefix,
        horizon,
    )

    memory_gpu_obj = memory_gpu_cached(
        name,
        horizon,
        (
            f"F{fold}_"
            f"prefix{prefix}"
        ),
        retriever,
        memory,
        C,
    )

    anchors = eval_anchors(
        prefix,
        oof_end,
        horizon,
        stride=
            OOF_ANCHOR_STRIDE,
    )

    print(
        f"OOF {name} H={horizon} F{fold}: "
        f"prefix={prefix}, "
        f"end={oof_end}, "
        f"anchors={len(anchors)}, "
        f"pairs={len(anchors)*C}, "
        f"memory/C={memory['M']}"
    )

    out = collect_gate_data(
        data,
        horizon,
        direct_model,
        retriever,
        memory_gpu_obj,
        z,
        anchors,
    )

    np.savez_compressed(
        out_path,
        **out,
    )

    del (
        direct_model,
        retriever,
        memory,
        memory_gpu_obj,
        prefix_scaler,
        z,
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return out


def build_validation_cache(
    data,
    horizon,
    direct_model,
    retriever,
):
    name = data[
        "name"
    ]

    C = data[
        "n_channels"
    ]

    path = val_cache_path(
        name,
        horizon,
    )

    if (
        path.exists()
        and RESUME
        and not FORCE
    ):
        obj = np.load(
            path
        )

        print(
            "Loaded validation cache:",
            path.name,
        )

        return {
            key:
                obj[
                    key
                ]
            for key in [
                "feature",
                "abc",
                "anchor",
                "channel",
            ]
        }

    memory = build_memory(
        data[
            "z"
        ],
        C,
        data[
            "train_end"
        ],
        horizon,
    )

    memory_gpu_obj = memory_gpu_cached(
        name,
        horizon,
        "validation_train_memory",
        retriever,
        memory,
        C,
    )

    anchors = eval_anchors(
        data[
            "train_end"
        ],
        data[
            "val_end"
        ],
        horizon,
        stride=1,
    )

    print(
        f"Validation {name} H={horizon}: "
        f"anchors={len(anchors)}, "
        f"pairs={len(anchors)*C}, "
        f"memory/C={memory['M']}"
    )

    out = collect_gate_data(
        data,
        horizon,
        direct_model,
        retriever,
        memory_gpu_obj,
        data[
            "z"
        ],
        anchors,
    )

    np.savez_compressed(
        path,
        **out,
    )

    del (
        memory,
        memory_gpu_obj,
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return out


## 16. Cross-fitted gate and validation-only calibration

In [17]:

def fit_feature_scaler(
    x,
):
    median = np.median(
        x,
        axis=0,
    ).astype(
        np.float32
    )

    q25 = np.percentile(
        x,
        25,
        axis=0,
    )

    q75 = np.percentile(
        x,
        75,
        axis=0,
    )

    iqr = (
        q75
        - q25
    ).astype(
        np.float32
    )

    iqr = np.where(
        iqr < 1e-5,
        1.0,
        iqr,
    ).astype(
        np.float32
    )

    return (
        median,
        iqr,
    )


def scale_features(
    x,
    median,
    iqr,
):
    return np.clip(
        (
            x
            - median
        )
        / iqr,
        -8.0,
        8.0,
    ).astype(
        np.float32
    )


def gate_loss(
    alpha,
    abc,
):
    return (
        abc[
            :,
            0
        ]
        + 2.0
        * alpha
        * abc[
            :,
            1
        ]
        + alpha
        * alpha
        * abc[
            :,
            2
        ]
    ).mean()


def gate_checkpoint_path(
    name,
    horizon,
):
    return (
        DIRS[
            "gate"
        ]
        / (
            f"{name}_H{horizon}_"
            "gate.pt"
        )
    )


def train_gate_epoch(
    model,
    optimizer,
    x,
    abc,
    rng,
):
    model.train()

    order = rng.permutation(
        len(
            x
        )
    )

    losses = []

    for i in range(
        0,
        len(
            order
        ),
        GATE_BATCH,
    ):
        ids = order[
            i:
            i+GATE_BATCH
        ]

        xt = torch.from_numpy(
            x[
                ids
            ]
        ).to(
            DEVICE
        )

        at = torch.from_numpy(
            abc[
                ids
            ]
        ).to(
            DEVICE
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        alpha = model(
            xt
        )

        loss = gate_loss(
            alpha,
            at,
        )

        loss.backward()
        optimizer.step()

        losses.append(
            float(
                loss.item()
            )
        )

        del (
            xt,
            at,
            alpha,
            loss,
        )

    return float(
        np.mean(
            losses
        )
    )


@torch.no_grad()
def evaluate_gate(
    model,
    x,
    abc,
):
    model.eval()

    total = 0.0
    n = 0
    alpha_sum = 0.0

    for i in range(
        0,
        len(
            x
        ),
        GATE_BATCH,
    ):
        xt = torch.from_numpy(
            x[
                i:
                i+GATE_BATCH
            ]
        ).to(
            DEVICE
        )

        at = torch.from_numpy(
            abc[
                i:
                i+GATE_BATCH
            ]
        ).to(
            DEVICE
        )

        alpha = model(
            xt
        )

        each = (
            at[
                :,
                0
            ]
            + 2.0
            * alpha
            * at[
                :,
                1
            ]
            + alpha
            * alpha
            * at[
                :,
                2
            ]
        )

        total += float(
            each.sum()
        )

        n += len(
            alpha
        )

        alpha_sum += float(
            alpha.sum()
        )

        del (
            xt,
            at,
            alpha,
            each,
        )

    return (
        total
        / n,
        alpha_sum
        / n,
    )


def train_crossfit_gate(
    name,
    horizon,
    oof_x,
    oof_abc,
    val_x,
    val_abc,
):
    path = gate_checkpoint_path(
        name,
        horizon,
    )

    if (
        path.exists()
        and RESUME
        and not FORCE
    ):
        ckpt = load_torch(
            path
        )

        model = CrossFitAdaptiveGate().to(
            DEVICE
        )

        model.load_state_dict(
            ckpt[
                "StateDict"
            ]
        )

        model.eval()

        print(
            "Loaded gate:",
            path.name,
        )

        return (
            model,
            ckpt,
        )

    median, iqr = fit_feature_scaler(
        oof_x
    )

    train_x = scale_features(
        oof_x,
        median,
        iqr,
    )

    valid_x = scale_features(
        val_x,
        median,
        iqr,
    )

    seed = (
        CROSSFIT_SEED
        + horizon
        * 3000
        + sum(
            map(
                ord,
                name,
            )
        )
    )

    set_seed(
        seed
    )

    model = CrossFitAdaptiveGate().to(
        DEVICE
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=GATE_LR,
        weight_decay=GATE_WD,
    )

    rng = np.random.default_rng(
        seed
        + 1
    )

    best = float(
        "inf"
    )

    best_epoch = -1
    wait = 0
    history = []

    for epoch in range(
        1,
        GATE_MAX_EPOCHS
        + 1,
    ):
        train_mse = train_gate_epoch(
            model,
            optimizer,
            train_x,
            oof_abc,
            rng,
        )

        val_mse, mean_alpha = evaluate_gate(
            model,
            valid_x,
            val_abc,
        )

        history.append({
            "Epoch":
                epoch,
            "OOFTrainMSE":
                train_mse,
            "ValMSE":
                val_mse,
            "ValMeanAlpha":
                mean_alpha,
        })

        if (
            val_mse
            < best
            - 1e-10
        ):
            best = val_mse
            best_epoch = epoch
            wait = 0
        else:
            wait += 1

        print(
            f"Gate {name:11s} H={horizon:3d} "
            f"ep={epoch:02d} "
            f"OOF={train_mse:.6f} "
            f"val={val_mse:.6f} "
            f"alpha={mean_alpha:.3f} "
            f"best={best:.6f}@{best_epoch}"
        )

        if (
            wait
            >= GATE_PATIENCE
        ):
            break

    # Reinitialize and fit only on OOF for the
    # validation-selected number of epochs.
    set_seed(
        seed
    )

    final = CrossFitAdaptiveGate().to(
        DEVICE
    )

    final_opt = torch.optim.AdamW(
        final.parameters(),
        lr=GATE_LR,
        weight_decay=GATE_WD,
    )

    final_rng = np.random.default_rng(
        seed
        + 2
    )

    for _ in range(
        best_epoch
    ):
        train_gate_epoch(
            final,
            final_opt,
            train_x,
            oof_abc,
            final_rng,
        )

    final.eval()

    ckpt = {
        "BestEpoch":
            best_epoch,
        "BestValMSE":
            best,
        "FeatureMedian":
            median,
        "FeatureIQR":
            iqr,
        "StateDict": {
            k:
                v.detach()
                .cpu()
                .clone()
            for k, v
            in final.state_dict().items()
        },
    }

    torch.save(
        ckpt,
        path,
    )

    pd.DataFrame(
        history
    ).to_csv(
        DIRS[
            "history"
        ]
        / (
            f"{name}_H{horizon}_"
            "gate_history.csv"
        ),
        index=False,
    )

    return (
        final,
        ckpt,
    )


def mse_scalar(
    abc,
    alpha,
):
    A = abc.astype(
        np.float64
    )

    x = float(
        alpha
    )

    return float(
        np.mean(
            A[
                :,
                0
            ]
            + 2.0
            * x
            * A[
                :,
                1
            ]
            + x
            * x
            * A[
                :,
                2
            ]
        )
    )


def choose_scalar(
    abc,
):
    rows = []

    best_alpha = None
    best_mse = float(
        "inf"
    )

    for alpha in ALPHA_GRID:
        mse = mse_scalar(
            abc,
            alpha,
        )

        rows.append({
            "Alpha":
                float(
                    alpha
                ),
            "MSE":
                mse,
        })

        if (
            mse
            < best_mse
            - 1e-10
        ):
            best_mse = mse
            best_alpha = float(
                alpha
            )

    return (
        best_alpha,
        pd.DataFrame(
            rows
        ),
    )


@torch.no_grad()
def gate_alpha(
    model,
    ckpt,
    x,
):
    sx = scale_features(
        x,
        ckpt[
            "FeatureMedian"
        ],
        ckpt[
            "FeatureIQR"
        ],
    )

    outputs = []

    for i in range(
        0,
        len(
            sx
        ),
        GATE_BATCH,
    ):
        t = torch.from_numpy(
            sx[
                i:
                i+GATE_BATCH
            ]
        ).to(
            DEVICE
        )

        outputs.append(
            model(
                t
            ).cpu()
            .numpy()
        )

        del t

    return np.concatenate(
        outputs
    ).astype(
        np.float32
    )


def mse_pair(
    abc,
    alpha,
):
    A = abc.astype(
        np.float64
    )

    x = np.asarray(
        alpha,
        dtype=np.float64,
    )

    return float(
        np.mean(
            A[
                :,
                0
            ]
            + 2.0
            * x
            * A[
                :,
                1
            ]
            + x
            * x
            * A[
                :,
                2
            ]
        )
    )


def choose_lambda(
    abc,
    gate_alpha_values,
    scalar_alpha,
):
    rows = []

    best_lambda = None
    best_mse = float(
        "inf"
    )

    for lmb in LAMBDA_GRID:
        alpha = (
            (
                1.0
                - float(
                    lmb
                )
            )
            * scalar_alpha
            + float(
                lmb
            )
            * gate_alpha_values
        )

        mse = mse_pair(
            abc,
            alpha,
        )

        rows.append({
            "Lambda":
                float(
                    lmb
                ),
            "MSE":
                mse,
            "MeanAlpha":
                float(
                    alpha.mean()
                ),
        })

        if (
            mse
            < best_mse
            - 1e-10
        ):
            best_mse = mse
            best_lambda = float(
                lmb
            )

    return (
        best_lambda,
        pd.DataFrame(
            rows
        ),
    )


## 17. Final test and Oracle diagnostic

In [18]:

def empty_stat():
    return {
        "sse":
            0.0,
        "sae":
            0.0,
        "n":
            0,
    }


def update_stat(
    stat,
    pred,
    true,
):
    e = (
        pred
        - true
    )

    stat[
        "sse"
    ] += float(
        (
            e
            * e
        ).sum()
    )

    stat[
        "sae"
    ] += float(
        e.abs().sum()
    )

    stat[
        "n"
    ] += e.numel()


def finish_stat(
    stat,
):
    return (
        stat[
            "sse"
        ]
        / stat[
            "n"
        ],
        stat[
            "sae"
        ]
        / stat[
            "n"
        ],
    )


def oracle_alpha_and_prediction(
    direct,
    retrieval,
    true,
):
    e = (
        direct
        - true
    )

    delta = (
        retrieval
        - direct
    )

    alpha = torch.clamp(
        -(
            e
            * delta
        ).sum(
            dim=1
        )
        / (
            (
                delta
                * delta
            ).sum(
                dim=1
            )
            + 1e-8
        ),
        0.0,
        1.0,
    )

    pred = (
        direct
        + alpha[
            :,
            None
        ]
        * delta
    )

    return (
        alpha,
        pred,
    )


@torch.no_grad()
def test_evaluate(
    data,
    horizon,
    direct_model,
    retriever,
    memory_gpu_obj,
    gate,
    gate_ckpt,
    scalar_alpha,
    shrink_lambda,
):
    name = data[
        "name"
    ]

    C = data[
        "n_channels"
    ]

    direct_block = DIRECT_ANCHOR_BLOCK[
        name
    ]

    ret_block = retrieval_anchor_batch(
        name
    )

    anchors = eval_anchors(
        data[
            "val_end"
        ],
        data[
            "test_end"
        ],
        horizon,
        stride=1,
    )

    keys = [
        "Direct",
        "Retrieval",
        "Scalar",
        "RawAdaptive",
        "ShrinkAdaptive",
        "Oracle",
    ]

    stats = {
        key:
            empty_stat()
        for key in keys
    }

    anchor_mse = {
        key:
            []
        for key in keys
    }

    channel_sse_direct = np.zeros(
        C,
        dtype=np.float64,
    )

    channel_sse_shrink = np.zeros(
        C,
        dtype=np.float64,
    )

    channel_count = np.zeros(
        C,
        dtype=np.int64,
    )

    raw_alpha_sum = 0.0
    shrink_alpha_sum = 0.0
    oracle_alpha_sum = 0.0
    oracle_positive = 0
    n_pairs = 0

    processed = 0

    for outer in range(
        0,
        len(
            anchors
        ),
        direct_block,
    ):
        a_big = anchors[
            outer:
            outer+direct_block
        ]

        direct_big, true_big = (
            direct_residual_block(
                direct_model,
                data[
                    "z"
                ],
                data[
                    "marks"
                ],
                a_big,
                horizon,
            )
        )

        direct_big = direct_big.permute(
            0,
            2,
            1,
        ).contiguous()

        true_big = true_big.permute(
            0,
            2,
            1,
        ).contiguous()

        # anchor-level MSE accumulators within this direct block
        block_anchor_sums = {
            key:
                torch.zeros(
                    len(
                        a_big
                    ),
                    device=DEVICE,
                    dtype=torch.float64,
                )
            for key in keys
        }

        for inner in range(
            0,
            len(
                a_big
            ),
            ret_block,
        ):
            a = a_big[
                inner:
                inner+ret_block
            ]

            A = len(
                a
            )

            pair_anchor = np.repeat(
                a,
                C,
            )

            pair_channel = np.tile(
                np.arange(
                    C,
                    dtype=np.int64,
                ),
                A,
            )

            r = retrieve(
                retriever,
                memory_gpu_obj,
                data[
                    "z"
                ],
                pair_anchor,
                pair_channel,
                horizon,
            )

            retrieval = r[
                "cand"
            ].mean(
                dim=1
            )

            direct = direct_big[
                inner:
                inner+A
            ].reshape(
                -1,
                horizon,
            )

            true = true_big[
                inner:
                inner+A
            ].reshape(
                -1,
                horizon,
            )

            scalar = (
                direct
                + scalar_alpha
                * (
                    retrieval
                    - direct
                )
            )

            features = gate_features(
                r,
                retrieval,
                direct,
            ).cpu().numpy().astype(
                np.float32
            )

            scaled = scale_features(
                features,
                gate_ckpt[
                    "FeatureMedian"
                ],
                gate_ckpt[
                    "FeatureIQR"
                ],
            )

            gate_alpha_values = gate(
                torch.from_numpy(
                    scaled
                ).to(
                    DEVICE
                )
            )

            shrink_alpha_values = (
                (
                    1.0
                    - shrink_lambda
                )
                * scalar_alpha
                + shrink_lambda
                * gate_alpha_values
            )

            raw_adaptive = (
                direct
                + gate_alpha_values[
                    :,
                    None
                ]
                * (
                    retrieval
                    - direct
                )
            )

            shrink_adaptive = (
                direct
                + shrink_alpha_values[
                    :,
                    None
                ]
                * (
                    retrieval
                    - direct
                )
            )

            (
                oracle_alpha,
                oracle,
            ) = oracle_alpha_and_prediction(
                direct,
                retrieval,
                true,
            )

            predictions = {
                "Direct":
                    direct,
                "Retrieval":
                    retrieval,
                "Scalar":
                    scalar,
                "RawAdaptive":
                    raw_adaptive,
                "ShrinkAdaptive":
                    shrink_adaptive,
                "Oracle":
                    oracle,
            }

            for key, pred in predictions.items():
                update_stat(
                    stats[
                        key
                    ],
                    pred,
                    true,
                )

                per_anchor = (
                    (
                        (
                            pred
                            - true
                        )
                        ** 2
                    )
                    .reshape(
                        A,
                        C,
                        horizon,
                    )
                    .mean(
                        dim=(
                            1,
                            2,
                        )
                    )
                    .double()
                )

                block_anchor_sums[
                    key
                ][
                    inner:
                    inner+A
                ] = per_anchor

            direct_e2 = (
                (
                    direct
                    - true
                )
                ** 2
            ).reshape(
                A,
                C,
                horizon,
            )

            shrink_e2 = (
                (
                    shrink_adaptive
                    - true
                )
                ** 2
            ).reshape(
                A,
                C,
                horizon,
            )

            channel_sse_direct += (
                direct_e2.sum(
                    dim=(
                        0,
                        2,
                    )
                ).cpu()
                .numpy()
            )

            channel_sse_shrink += (
                shrink_e2.sum(
                    dim=(
                        0,
                        2,
                    )
                ).cpu()
                .numpy()
            )

            channel_count += (
                A
                * horizon
            )

            raw_alpha_sum += float(
                gate_alpha_values.sum()
            )

            shrink_alpha_sum += float(
                shrink_alpha_values.sum()
            )

            oracle_alpha_sum += float(
                oracle_alpha.sum()
            )

            oracle_positive += int(
                (
                    oracle_alpha
                    > 0.01
                ).sum()
            )

            n_pairs += len(
                gate_alpha_values
            )

            del (
                r,
                retrieval,
                direct,
                true,
                scalar,
                features,
                scaled,
                gate_alpha_values,
                shrink_alpha_values,
                raw_adaptive,
                shrink_adaptive,
                oracle_alpha,
                oracle,
                predictions,
                direct_e2,
                shrink_e2,
            )

        for key in keys:
            anchor_mse[
                key
            ].extend(
                block_anchor_sums[
                    key
                ].cpu()
                .numpy()
                .astype(
                    np.float32
                )
                .tolist()
            )

        processed += len(
            a_big
        )

        if (
            processed
            == len(
                a_big
            )
            or processed
            % 500
            < len(
                a_big
            )
            or processed
            == len(
                anchors
            )
        ):
            print(
                f"  test anchors "
                f"{processed}/{len(anchors)}"
            )

        del (
            direct_big,
            true_big,
            block_anchor_sums,
        )

    return {
        "anchors":
            anchors,
        "metrics": {
            key:
                finish_stat(
                    value
                )
            for key, value
            in stats.items()
        },
        "anchor_mse": {
            key:
                np.asarray(
                    value,
                    dtype=np.float32,
                )
            for key, value
            in anchor_mse.items()
        },
        "channel_direct_mse":
            channel_sse_direct
            / channel_count,
        "channel_shrink_mse":
            channel_sse_shrink
            / channel_count,
        "raw_mean_alpha":
            raw_alpha_sum
            / n_pairs,
        "shrink_mean_alpha":
            shrink_alpha_sum
            / n_pairs,
        "oracle_mean_alpha":
            oracle_alpha_sum
            / n_pairs,
        "oracle_positive_fraction":
            oracle_positive
            / n_pairs,
    }


## 18. Paired moving-block bootstrap

In [19]:

def moving_block_bootstrap(
    difference,
    n_boot=5000,
    block=24,
    seed=222222,
):
    x = np.asarray(
        difference,
        dtype=np.float64,
    )

    n = len(
        x
    )

    L = min(
        block,
        n,
    )

    rng = np.random.default_rng(
        seed
    )

    n_blocks = int(
        np.ceil(
            n
            / L
        )
    )

    max_start = max(
        1,
        n
        - L
        + 1,
    )

    boot = np.empty(
        n_boot,
        dtype=np.float64,
    )

    for b in range(
        n_boot
    ):
        starts = rng.integers(
            0,
            max_start,
            size=n_blocks,
        )

        sample = np.concatenate(
            [
                x[
                    s:
                    s+L
                ]
                for s in starts
            ]
        )[
            :n
        ]

        boot[
            b
        ] = sample.mean()

    return {
        "MeanImprovement":
            float(
                x.mean()
            ),
        "CI_Low":
            float(
                np.quantile(
                    boot,
                    0.025,
                )
            ),
        "CI_High":
            float(
                np.quantile(
                    boot,
                    0.975,
                )
            ),
    }



# 19. Main Experiment

각 dataset × horizon에서 다음을 수행합니다.

1. Experiment 21 full iTransformer load
2. Experiment 13 frozen full retriever load
3. three chronological OOF folds
4. cross-fitted gate
5. validation scalar / shrinkage calibration
6. static train+validation retrieval memory
7. all-window test
8. Oracle diagnostic
9. moving-block bootstrap
10. per-channel diagnostic
11. condition별 즉시 저장

중간 결과를 본 뒤 method를 변경하지 않습니다.


In [20]:

SUMMARY_PATH = (
    ROOT
    / "summary.csv"
)

BOOTSTRAP_PATH = (
    ROOT
    / "bootstrap.csv"
)

FOLD_PATH = (
    ROOT
    / "folds.csv"
)

existing = (
    pd.read_csv(
        SUMMARY_PATH
    )
    if (
        RESUME
        and SUMMARY_PATH.exists()
    )
    else pd.DataFrame()
)

summary_rows = (
    existing.to_dict(
        "records"
    )
    if len(
        existing
    )
    else []
)

bootstrap_rows = (
    pd.read_csv(
        BOOTSTRAP_PATH
    ).to_dict(
        "records"
    )
    if (
        RESUME
        and BOOTSTRAP_PATH.exists()
    )
    else []
)

fold_rows = (
    pd.read_csv(
        FOLD_PATH
    ).to_dict(
        "records"
    )
    if (
        RESUME
        and FOLD_PATH.exists()
    )
    else []
)


def already_done(
    name,
    horizon,
):
    if not len(
        existing
    ):
        return False

    return bool(
        (
            (
                existing[
                    "Dataset"
                ]
                == name
            )
            & (
                existing[
                    "Horizon"
                ]
                == horizon
            )
        ).any()
    )


for name, horizon in TASKS:
    if already_done(
        name,
        horizon,
    ):
        print(
            f"SKIP completed: "
            f"{name} H={horizon}"
        )
        continue

    start_time = time.time()

    data = DATA[
        name
    ]

    C = data[
        "n_channels"
    ]

    print(
        "\n"
        + "#"
        * 150
    )

    print(
        f"{name} | H={horizon} | "
        "iTransformer + FROZEN HISTORICAL MEMORY"
    )

    print(
        "#"
        * 150
    )

    # --------------------------------------------------------
    # 1. Frozen strong full direct.
    # --------------------------------------------------------
    direct_model, direct_ckpt = load_exp21_direct(
        name,
        horizon,
    )

    ref = exp21_reference(
        name,
        horizon,
    )

    direct_epochs = int(
        direct_ckpt[
            "BestEpoch"
        ]
    )

    print(
        f"Experiment 21 direct: "
        f"MSE={float(ref['Test_MSE']):.6f}, "
        f"MAE={float(ref['Test_MAE']):.6f}, "
        f"best_epoch={direct_epochs}"
    )

    # --------------------------------------------------------
    # 2. Frozen full retriever.
    # --------------------------------------------------------
    retriever, retriever_ckpt = load_frozen_retriever(
        full_retriever_ckpt_path(
            name,
            horizon,
        )
    )

    # --------------------------------------------------------
    # 3. OOF cross-fitting.
    # --------------------------------------------------------
    oof_parts = []

    for fold, (
        p0,
        p1,
    ) in enumerate(
        FOLDS,
        start=1,
    ):
        part = build_oof_fold(
            data,
            horizon,
            fold,
            p0,
            p1,
            direct_epochs,
        )

        oof_parts.append(
            part
        )

        fold_rows = [
            r
            for r in fold_rows
            if not (
                r.get(
                    "Dataset"
                )
                == name
                and int(
                    r.get(
                        "Horizon",
                        -1,
                    )
                )
                == horizon
                and int(
                    r.get(
                        "Fold",
                        -1,
                    )
                )
                == fold
            )
        ]

        fold_rows.append({
            "Dataset":
                name,
            "Horizon":
                horizon,
            "Fold":
                fold,
            "PrefixFrac":
                p0,
            "OOFEndFrac":
                p1,
            "Pairs":
                len(
                    part[
                        "feature"
                    ]
                ),
            "Anchors":
                len(
                    np.unique(
                        part[
                            "anchor"
                        ]
                    )
                ),
            "DirectFixedEpochs":
                direct_epochs,
            "RetrieverCheckpoint":
                str(
                    fold_retriever_ckpt_path(
                        name,
                        horizon,
                        fold,
                    )
                ),
        })

        pd.DataFrame(
            fold_rows
        ).to_csv(
            FOLD_PATH,
            index=False,
        )

    oof_x = np.concatenate(
        [
            p[
                "feature"
            ]
            for p in oof_parts
        ],
        axis=0,
    )

    oof_abc = np.concatenate(
        [
            p[
                "abc"
            ]
            for p in oof_parts
        ],
        axis=0,
    )

    print(
        "Total OOF pairs:",
        len(
            oof_x
        ),
    )

    # --------------------------------------------------------
    # 4. Validation.
    # --------------------------------------------------------
    val_data = build_validation_cache(
        data,
        horizon,
        direct_model,
        retriever,
    )

    gate, gate_ckpt = train_crossfit_gate(
        name,
        horizon,
        oof_x,
        oof_abc,
        val_data[
            "feature"
        ],
        val_data[
            "abc"
        ],
    )

    scalar_alpha, scalar_curve = choose_scalar(
        val_data[
            "abc"
        ]
    )

    raw_val_alpha = gate_alpha(
        gate,
        gate_ckpt,
        val_data[
            "feature"
        ],
    )

    shrink_lambda, lambda_curve = choose_lambda(
        val_data[
            "abc"
        ],
        raw_val_alpha,
        scalar_alpha,
    )

    scalar_curve.to_csv(
        DIRS[
            "calibration"
        ]
        / (
            f"{name}_H{horizon}_"
            "scalar_alpha.csv"
        ),
        index=False,
    )

    lambda_curve.to_csv(
        DIRS[
            "calibration"
        ]
        / (
            f"{name}_H{horizon}_"
            "shrink_lambda.csv"
        ),
        index=False,
    )

    print(
        f"Validation calibration | "
        f"alpha0={scalar_alpha:.2f} | "
        f"lambda={shrink_lambda:.2f} | "
        f"gateEpoch={gate_ckpt['BestEpoch']}"
    )

    del (
        oof_x,
        oof_abc,
        oof_parts,
        raw_val_alpha,
        val_data,
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # --------------------------------------------------------
    # 5. Static test retrieval memory: train + validation only.
    # --------------------------------------------------------
    test_memory = build_memory(
        data[
            "z"
        ],
        C,
        data[
            "val_end"
        ],
        horizon,
    )

    test_memory_gpu = memory_gpu_cached(
        name,
        horizon,
        "test_trainval_memory",
        retriever,
        test_memory,
        C,
    )

    test = test_evaluate(
        data,
        horizon,
        direct_model,
        retriever,
        test_memory_gpu,
        gate,
        gate_ckpt,
        scalar_alpha,
        shrink_lambda,
    )

    metrics = test[
        "metrics"
    ]

    (
        direct_mse,
        direct_mae,
    ) = metrics[
        "Direct"
    ]

    (
        retrieval_mse,
        retrieval_mae,
    ) = metrics[
        "Retrieval"
    ]

    (
        scalar_mse,
        scalar_mae,
    ) = metrics[
        "Scalar"
    ]

    (
        raw_mse,
        raw_mae,
    ) = metrics[
        "RawAdaptive"
    ]

    (
        shrink_mse,
        shrink_mae,
    ) = metrics[
        "ShrinkAdaptive"
    ]

    (
        oracle_mse,
        oracle_mae,
    ) = metrics[
        "Oracle"
    ]

    exp21_mse = float(
        ref[
            "Test_MSE"
        ]
    )

    exp21_mae = float(
        ref[
            "Test_MAE"
        ]
    )

    parity_diff = abs(
        direct_mse
        - exp21_mse
    )

    if (
        parity_diff
        >= DIRECT_PARITY_TOL
    ):
        raise RuntimeError(
            f"Final direct parity failed for "
            f"{name} H={horizon}: "
            f"{parity_diff}"
        )

    # --------------------------------------------------------
    # 6. Paired bootstrap.
    # --------------------------------------------------------
    bootstrap_rows = [
        r
        for r in bootstrap_rows
        if not (
            r.get(
                "Dataset"
            )
            == name
            and int(
                r.get(
                    "Horizon",
                    -1,
                )
            )
            == horizon
        )
    ]

    b_direct = moving_block_bootstrap(
        test[
            "anchor_mse"
        ][
            "Direct"
        ]
        - test[
            "anchor_mse"
        ][
            "ShrinkAdaptive"
        ],
        seed=
            222222
            + horizon
            + sum(
                map(
                    ord,
                    name,
                )
            ),
    )

    bootstrap_rows.append({
        "Dataset":
            name,
        "Horizon":
            horizon,
        "Comparison":
            "Direct-ShrinkAdaptive",
        **b_direct,
        "SignificantPositive":
            b_direct[
                "CI_Low"
            ]
            > 0.0,
    })

    b_scalar = moving_block_bootstrap(
        test[
            "anchor_mse"
        ][
            "Scalar"
        ]
        - test[
            "anchor_mse"
        ][
            "ShrinkAdaptive"
        ],
        seed=
            333333
            + horizon
            + sum(
                map(
                    ord,
                    name,
                )
            ),
    )

    bootstrap_rows.append({
        "Dataset":
            name,
        "Horizon":
            horizon,
        "Comparison":
            "Scalar-ShrinkAdaptive",
        **b_scalar,
        "SignificantPositive":
            b_scalar[
                "CI_Low"
            ]
            > 0.0,
    })

    pd.DataFrame(
        bootstrap_rows
    ).to_csv(
        BOOTSTRAP_PATH,
        index=False,
    )

    # --------------------------------------------------------
    # 7. Per-channel diagnostic.
    # --------------------------------------------------------
    channel_df = pd.DataFrame({
        "ChannelIndex":
            np.arange(
                C
            ),
        "ChannelName":
            data[
                "columns"
            ],
        "Direct_MSE":
            test[
                "channel_direct_mse"
            ],
        "ShrinkAdaptive_MSE":
            test[
                "channel_shrink_mse"
            ],
    })

    channel_df[
        "Improvement"
    ] = (
        channel_df[
            "Direct_MSE"
        ]
        - channel_df[
            "ShrinkAdaptive_MSE"
        ]
    )

    channel_df[
        "Improvement_pct"
    ] = (
        100.0
        * channel_df[
            "Improvement"
        ]
        / channel_df[
            "Direct_MSE"
        ]
    )

    channel_df.to_csv(
        DIRS[
            "channel"
        ]
        / (
            f"{name}_H{horizon}_"
            "channel_mse.csv"
        ),
        index=False,
    )

    improved_channel_fraction = float(
        (
            channel_df[
                "Improvement"
            ]
            > 0.0
        ).mean()
    )

    # --------------------------------------------------------
    # 8. Save anchor-level paired losses.
    # --------------------------------------------------------
    np.savez_compressed(
        DIRS[
            "paired"
        ]
        / (
            f"{name}_H{horizon}_"
            "anchor_mse.npz"
        ),
        TestAnchors=
            test[
                "anchors"
            ],
        Direct=
            test[
                "anchor_mse"
            ][
                "Direct"
            ],
        Retrieval=
            test[
                "anchor_mse"
            ][
                "Retrieval"
            ],
        Scalar=
            test[
                "anchor_mse"
            ][
                "Scalar"
            ],
        RawAdaptive=
            test[
                "anchor_mse"
            ][
                "RawAdaptive"
            ],
        ShrinkAdaptive=
            test[
                "anchor_mse"
            ][
                "ShrinkAdaptive"
            ],
        Oracle=
            test[
                "anchor_mse"
            ][
                "Oracle"
            ],
    )

    # --------------------------------------------------------
    # 9. Final summary.
    # --------------------------------------------------------
    row = {
        "Dataset":
            name,
        "Horizon":
            horizon,
        "Channels":
            C,
        "iTransformer_MSE":
            direct_mse,
        "iTransformer_MAE":
            direct_mae,
        "Experiment21Reference_MSE":
            exp21_mse,
        "Experiment21Reference_MAE":
            exp21_mae,
        "DirectParityAbsDiff":
            parity_diff,
        "Retrieval_MSE":
            retrieval_mse,
        "Retrieval_MAE":
            retrieval_mae,
        "ScalarAlpha":
            scalar_alpha,
        "Scalar_MSE":
            scalar_mse,
        "Scalar_MAE":
            scalar_mae,
        "RawAdaptive_MSE":
            raw_mse,
        "RawAdaptive_MAE":
            raw_mae,
        "ShrinkLambda":
            shrink_lambda,
        "ShrinkAdaptive_MSE":
            shrink_mse,
        "ShrinkAdaptive_MAE":
            shrink_mae,
        "Oracle_MSE":
            oracle_mse,
        "Oracle_MAE":
            oracle_mae,
        "RawMeanAlpha":
            test[
                "raw_mean_alpha"
            ],
        "ShrinkMeanAlpha":
            test[
                "shrink_mean_alpha"
            ],
        "OracleMeanAlpha":
            test[
                "oracle_mean_alpha"
            ],
        "OraclePositiveFraction":
            test[
                "oracle_positive_fraction"
            ],
        "ImprovedChannelFraction":
            improved_channel_fraction,
        "ScalarGainVsDirect_pct":
            (
                100.0
                * (
                    direct_mse
                    - scalar_mse
                )
                / direct_mse
            ),
        "ShrinkGainVsDirect_pct":
            (
                100.0
                * (
                    direct_mse
                    - shrink_mse
                )
                / direct_mse
            ),
        "ShrinkGainVsScalar_pct":
            (
                100.0
                * (
                    scalar_mse
                    - shrink_mse
                )
                / scalar_mse
            ),
        "OracleHeadroomFromDirect_pct":
            (
                100.0
                * (
                    direct_mse
                    - oracle_mse
                )
                / direct_mse
            ),
        "OracleHeadroomFromShrink_pct":
            (
                100.0
                * (
                    shrink_mse
                    - oracle_mse
                )
                / shrink_mse
            ),
        "DirectBestEpoch":
            direct_epochs,
        "FrozenRetrieverBestEpoch":
            retriever_ckpt.get(
                "BestEpoch",
                np.nan,
            ),
        "GateBestEpoch":
            int(
                gate_ckpt[
                    "BestEpoch"
                ]
            ),
        "TestMemoryPerChannel":
            int(
                test_memory[
                    "M"
                ]
            ),
        "TestWindows":
            len(
                test[
                    "anchors"
                ]
            ),
        "RuntimeMinutes":
            (
                time.time()
                - start_time
            )
            / 60.0,
    }

    summary_rows = [
        r
        for r in summary_rows
        if not (
            r.get(
                "Dataset"
            )
            == name
            and int(
                r.get(
                    "Horizon",
                    -1,
                )
            )
            == horizon
        )
    ]

    summary_rows.append(
        row
    )

    pd.DataFrame(
        summary_rows
    ).to_csv(
        SUMMARY_PATH,
        index=False,
    )

    print(
        "\nFINAL CONDITION RESULT"
    )

    display(
        pd.DataFrame([
            row
        ])[
            [
                "Dataset",
                "Horizon",
                "iTransformer_MSE",
                "ShrinkAdaptive_MSE",
                "ShrinkGainVsDirect_pct",
                "iTransformer_MAE",
                "ShrinkAdaptive_MAE",
                "Retrieval_MSE",
                "Scalar_MSE",
                "RawAdaptive_MSE",
                "Oracle_MSE",
                "ScalarAlpha",
                "ShrinkLambda",
                "ShrinkMeanAlpha",
                "ImprovedChannelFraction",
                "OracleHeadroomFromDirect_pct",
                "RuntimeMinutes",
            ]
        ]
    )

    del (
        direct_model,
        direct_ckpt,
        retriever,
        retriever_ckpt,
        gate,
        gate_ckpt,
        test_memory,
        test_memory_gpu,
        test,
        channel_df,
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


summary_df = (
    pd.DataFrame(
        summary_rows
    )
    .sort_values(
        [
            "Dataset",
            "Horizon",
        ]
    )
    .reset_index(
        drop=True
    )
)

display(
    summary_df
)



######################################################################################################################################################
ETTm1 | H=96 | iTransformer + FROZEN HISTORICAL MEMORY
######################################################################################################################################################
Experiment 21 direct: MSE=0.336712, MAE=0.373422, best_epoch=1
Updating learning rate to 0.0001
Fold direct ETTm1       H= 96 F1 ep=01/1 train=0.340237 lr=1.000e-04
Building memory embedding: ETTm1_H96_F1_prefix19008_emb.npy (7, 785, 64)
  channel 1/7
  channel 7/7
OOF ETTm1 H=96 F1: prefix=19008, end=24192, anchors=1273, pairs=8911, memory/C=785
Updating learning rate to 0.0001
Fold direct ETTm1       H= 96 F2 ep=01/1 train=0.304745 lr=1.000e-04
Building memory embedding: ETTm1_H96_F2_prefix24192_emb.npy (7, 1001, 64)
  channel 1/7
  channel 7/7
OOF ETTm1 H=96 F2: prefix=24192, end=29376, anchors=1273, pairs=8911, memory/C=1001
Updat

,Dataset,Horizon,iTransformer_MSE,ShrinkAdaptive_MSE,ShrinkGainVsDirect_pct,iTransformer_MAE,ShrinkAdaptive_MAE,Retrieval_MSE,Scalar_MSE,RawAdaptive_MSE,Oracle_MSE,ScalarAlpha,ShrinkLambda,ShrinkMeanAlpha,ImprovedChannelFraction,OracleHeadroomFromDirect_pct,RuntimeMinutes
0,ETTm1,96,0.336712,0.337092,-0.112813,0.373422,0.373635,1.273082,0.336712,0.337788,0.280151,0.0,0.75,0.071921,0.428571,16.797973,0.390909



######################################################################################################################################################
ETTm1 | H=192 | iTransformer + FROZEN HISTORICAL MEMORY
######################################################################################################################################################
Experiment 21 direct: MSE=0.394380, MAE=0.402333, best_epoch=3
Updating learning rate to 0.0001
Fold direct ETTm1       H=192 F1 ep=01/3 train=0.401970 lr=1.000e-04
Updating learning rate to 5e-05
Fold direct ETTm1       H=192 F1 ep=02/3 train=0.354952 lr=5.000e-05
Updating learning rate to 2.5e-05
Fold direct ETTm1       H=192 F1 ep=03/3 train=0.325659 lr=2.500e-05
Building memory embedding: ETTm1_H192_F1_prefix19008_emb.npy (7, 781, 64)
  channel 1/7
  channel 7/7
OOF ETTm1 H=192 F1: prefix=19008, end=24192, anchors=1249, pairs=8743, memory/C=781
Updating learning rate to 0.0001
Fold direct ETTm1       H=192 F2 ep=01/3 train=0.3529

,Dataset,Horizon,iTransformer_MSE,ShrinkAdaptive_MSE,ShrinkGainVsDirect_pct,iTransformer_MAE,ShrinkAdaptive_MAE,Retrieval_MSE,Scalar_MSE,RawAdaptive_MSE,Oracle_MSE,ScalarAlpha,ShrinkLambda,ShrinkMeanAlpha,ImprovedChannelFraction,OracleHeadroomFromDirect_pct,RuntimeMinutes
0,ETTm1,192,0.39438,0.39062,0.953379,0.402333,0.400364,1.32552,0.392679,0.39062,0.328181,0.1,1.0,0.084481,1.0,16.785594,0.897363



######################################################################################################################################################
ETTm1 | H=336 | iTransformer + FROZEN HISTORICAL MEMORY
######################################################################################################################################################
Experiment 21 direct: MSE=0.424327, MAE=0.422129, best_epoch=2
Updating learning rate to 0.0001
Fold direct ETTm1       H=336 F1 ep=01/2 train=0.485104 lr=1.000e-04
Updating learning rate to 5e-05
Fold direct ETTm1       H=336 F1 ep=02/2 train=0.433559 lr=5.000e-05
Building memory embedding: ETTm1_H336_F1_prefix19008_emb.npy (7, 775, 64)
  channel 1/7
  channel 7/7
OOF ETTm1 H=336 F1: prefix=19008, end=24192, anchors=1213, pairs=8491, memory/C=775
Updating learning rate to 0.0001
Fold direct ETTm1       H=336 F2 ep=01/2 train=0.413914 lr=1.000e-04
Updating learning rate to 5e-05
Fold direct ETTm1       H=336 F2 ep=02/2 train=0.373735

...


######################################################################################################################################################
Electricity | H=336 | iTransformer + FROZEN HISTORICAL MEMORY
######################################################################################################################################################
Experiment 21 direct: MSE=0.179687, MAE=0.272384, best_epoch=9
Updating learning rate to 0.0005
Fold direct Electricity H=336 F1 ep=01/9 train=0.241576 lr=5.000e-04
Updating learning rate to 0.00025
Fold direct Electricity H=336 F1 ep=02/9 train=0.209079 lr=2.500e-04
Updating learning rate to 0.000125
Fold direct Electricity H=336 F1 ep=03/9 train=0.197080 lr=1.250e-04
Updating learning rate to 6.25e-05
Fold direct Electricity H=336 F1 ep=04/9 train=0.191694 lr=6.250e-05
Updating learning rate to 3.125e-05
Fold direct Electricity H=336 F1 ep=05/9 train=0.188226 lr=3.125e-05
Updating learning rate to 1.5625e-05
Fold direct Elect

,Dataset,Horizon,iTransformer_MSE,ShrinkAdaptive_MSE,ShrinkGainVsDirect_pct,iTransformer_MAE,ShrinkAdaptive_MAE,Retrieval_MSE,Scalar_MSE,RawAdaptive_MSE,Oracle_MSE,ScalarAlpha,ShrinkLambda,ShrinkMeanAlpha,ImprovedChannelFraction,OracleHeadroomFromDirect_pct,RuntimeMinutes
0,Electricity,336,0.179687,0.178503,0.659223,0.272384,0.273237,1.936648,0.179687,0.178503,0.166097,0.0,1.0,0.036784,0.809969,7.563408,10.112683



######################################################################################################################################################
Electricity | H=720 | iTransformer + FROZEN HISTORICAL MEMORY
######################################################################################################################################################
Experiment 21 direct: MSE=0.217054, MAE=0.304202, best_epoch=3
Updating learning rate to 0.0005
Fold direct Electricity H=720 F1 ep=01/3 train=0.292047 lr=5.000e-04
Updating learning rate to 0.00025
Fold direct Electricity H=720 F1 ep=02/3 train=0.256062 lr=2.500e-04
Updating learning rate to 0.000125
Fold direct Electricity H=720 F1 ep=03/3 train=0.239109 lr=1.250e-04
Building memory embedding: Electricity_H720_F1_prefix10126_emb.npy (321, 388, 64)
  channel 1/321
  channel 50/321
  channel 100/321
  channel 150/321
  channel 200/321
  channel 250/321
  channel 300/321
  channel 321/321
OOF Electricity H=720 F1: prefix=10126, 

,Dataset,Horizon,iTransformer_MSE,ShrinkAdaptive_MSE,ShrinkGainVsDirect_pct,iTransformer_MAE,ShrinkAdaptive_MAE,Retrieval_MSE,Scalar_MSE,RawAdaptive_MSE,Oracle_MSE,ScalarAlpha,ShrinkLambda,ShrinkMeanAlpha,ImprovedChannelFraction,OracleHeadroomFromDirect_pct,RuntimeMinutes
0,Electricity,720,0.217054,0.215217,0.845958,0.304202,0.305317,1.898134,0.217054,0.215217,0.19811,0.0,1.0,0.045871,0.741433,8.727727,4.591641


,Dataset,Horizon,Channels,iTransformer_MSE,iTransformer_MAE,Experiment21Reference_MSE,Experiment21Reference_MAE,DirectParityAbsDiff,Retrieval_MSE,Retrieval_MAE,ScalarAlpha,Scalar_MSE,Scalar_MAE,RawAdaptive_MSE,RawAdaptive_MAE,ShrinkLambda,ShrinkAdaptive_MSE,ShrinkAdaptive_MAE,Oracle_MSE,Oracle_MAE,RawMeanAlpha,ShrinkMeanAlpha,OracleMeanAlpha,OraclePositiveFraction,ImprovedChannelFraction,ScalarGainVsDirect_pct,ShrinkGainVsDirect_pct,ShrinkGainVsScalar_pct,OracleHeadroomFromDirect_pct,OracleHeadroomFromShrink_pct,DirectBestEpoch,FrozenRetrieverBestEpoch,GateBestEpoch,TestMemoryPerChannel,TestWindows,RuntimeMinutes
0,ETTh1,96,7,0.392303,0.407334,0.392304,0.407334,1.073240e-07,1.799798,0.845403,0.1,0.403233,0.418905,0.394281,0.410792,1.00,0.394281,0.410792,0.362519,0.392469,0.079177,0.079177,0.161444,0.558348,0.571429,-2.786079,-0.504197,2.220029,7.592273,8.055853,1,2,30,473,2785,0.113245
1,ETTh1,192,7,0.442639,0.434737,0.442639,0.434737,3.424557e-08,1.791403,0.866270,0.1,0.453672,0.446403,0.449096,0.442669,1.00,0.449096,0.442669,0.413171,0.420761,0.089948,0.089948,0.154008,0.539659,0.428571,-2.492544,-1.458748,1.008655,6.657512,7.999566,1,5,19,469,2689,0.109456
2,ETTh1,336,7,0.489263,0.461457,0.489263,0.461457,3.946080e-09,1.850616,0.889907,0.1,0.498583,0.473796,0.498000,0.473326,1.00,0.498000,0.473326,0.454354,0.448419,0.098503,0.098503,0.152534,0.534718,0.571429,-1.904970,-1.785874,0.116870,7.135021,8.764374,1,3,6,463,2545,0.109588
3,ETTh1,720,7,0.506507,0.492894,0.506507,0.492894,1.634114e-08,1.920954,0.922036,0.2,0.560588,0.528789,0.519138,0.505294,0.50,0.536269,0.515912,0.471712,0.476156,0.099917,0.149958,0.143740,0.558009,0.571429,-10.677292,-5.876001,4.338099,6.869664,12.038295,1,3,1,447,2161,0.109269
4,ETTm1,96,7,0.336712,0.373422,0.336712,0.373422,9.791633e-09,1.273082,0.700499,0.0,0.336712,0.373422,0.337788,0.374302,0.75,0.337092,0.373635,0.280151,0.339608,0.095895,0.071921,0.211982,0.510772,0.428571,0.000000,-0.112813,-0.112813,16.797973,16.891730,1,2,16,1913,11425,0.390909
5,ETTm1,192,7,0.394380,0.402333,0.394380,0.402333,4.346991e-08,1.325520,0.731763,0.1,0.392679,0.402721,0.390620,0.400364,1.00,0.390620,0.400364,0.328181,0.368165,0.084481,0.084481,0.209284,0.565199,1.000000,0.431246,0.953379,0.524394,16.785594,15.984609,3,1,9,1909,11329,0.897363
6,ETTm1,336,7,0.424327,0.422129,0.424327,0.422129,3.578755e-08,1.364253,0.761082,0.1,0.424660,0.424178,0.425539,0.422317,0.75,0.423381,0.421551,0.368539,0.393009,0.114420,0.110815,0.180624,0.544505,0.714286,-0.078535,0.222913,0.301212,13.147345,12.953306,2,5,46,1903,11185,0.676535
7,ETTm1,720,7,0.496095,0.461646,0.496095,0.461646,1.632701e-08,1.424698,0.797972,0.1,0.489097,0.460496,0.489012,0.459917,0.75,0.488999,0.460045,0.427392,0.428242,0.094551,0.095913,0.185172,0.553031,1.000000,1.410598,1.430322,0.020006,13.848778,12.598657,3,5,5,1887,10801,0.994463
8,Electricity,96,321,0.148275,0.239707,0.148275,0.239707,4.570984e-09,1.839673,1.024234,0.0,0.148275,0.239707,0.146310,0.239500,1.00,0.146310,0.239500,0.134724,0.232317,0.031767,0.031767,0.065458,0.481328,0.813084,0.000000,1.325576,1.325576,9.139225,7.918617,7,7,46,869,5165,7.553352
9,Electricity,192,321,0.165101,0.256106,0.165100,0.256106,1.342250e-08,1.996508,1.058647,0.0,0.165101,0.256106,0.164833,0.257408,1.00,0.164833,0.257408,0.153132,0.249967,0.043048,0.043048,0.063306,0.466086,0.666667,0.000000,0.161767,0.161767,7.249021,7.098738,7,16,21,865,5069,7.628455


## 20. Compact evidence table

In [21]:

if not len(
    summary_df
):
    raise RuntimeError(
        "No completed Experiment 22 result."
    )

compact = summary_df[
    [
        "Dataset",
        "Horizon",
        "iTransformer_MSE",
        "ShrinkAdaptive_MSE",
        "ShrinkGainVsDirect_pct",
        "iTransformer_MAE",
        "ShrinkAdaptive_MAE",
        "ScalarAlpha",
        "ShrinkLambda",
        "ShrinkMeanAlpha",
        "ImprovedChannelFraction",
        "Oracle_MSE",
        "OracleHeadroomFromDirect_pct",
        "DirectParityAbsDiff",
    ]
].copy()

compact[
    "Winner"
] = np.where(
    compact[
        "ShrinkAdaptive_MSE"
    ]
    < compact[
        "iTransformer_MSE"
    ],
    "Ours",
    "Direct",
)

display(
    compact
)

compact.to_csv(
    ROOT
    / "compact_results.csv",
    index=False,
)

if BOOTSTRAP_PATH.exists():
    boot_df = pd.read_csv(
        BOOTSTRAP_PATH
    )

    direct_boot = boot_df[
        boot_df[
            "Comparison"
        ]
        == "Direct-ShrinkAdaptive"
    ].copy()

    direct_boot[
        "Significance"
    ] = np.select(
        [
            direct_boot[
                "CI_Low"
            ]
            > 0.0,
            direct_boot[
                "CI_High"
            ]
            < 0.0,
        ],
        [
            "Ours significantly better",
            "Direct significantly better",
        ],
        default=
            "Not significant",
    )

    display(
        direct_boot.sort_values(
            [
                "Dataset",
                "Horizon",
            ]
        )
    )


,Dataset,Horizon,iTransformer_MSE,ShrinkAdaptive_MSE,ShrinkGainVsDirect_pct,iTransformer_MAE,ShrinkAdaptive_MAE,ScalarAlpha,ShrinkLambda,ShrinkMeanAlpha,ImprovedChannelFraction,Oracle_MSE,OracleHeadroomFromDirect_pct,DirectParityAbsDiff,Winner
0,ETTh1,96,0.392303,0.394281,-0.504197,0.407334,0.410792,0.1,1.00,0.079177,0.571429,0.362519,7.592273,1.073240e-07,Direct
1,ETTh1,192,0.442639,0.449096,-1.458748,0.434737,0.442669,0.1,1.00,0.089948,0.428571,0.413171,6.657512,3.424557e-08,Direct
2,ETTh1,336,0.489263,0.498000,-1.785874,0.461457,0.473326,0.1,1.00,0.098503,0.571429,0.454354,7.135021,3.946080e-09,Direct
3,ETTh1,720,0.506507,0.536269,-5.876001,0.492894,0.515912,0.2,0.50,0.149958,0.571429,0.471712,6.869664,1.634114e-08,Direct
4,ETTm1,96,0.336712,0.337092,-0.112813,0.373422,0.373635,0.0,0.75,0.071921,0.428571,0.280151,16.797973,9.791633e-09,Direct
5,ETTm1,192,0.394380,0.390620,0.953379,0.402333,0.400364,0.1,1.00,0.084481,1.000000,0.328181,16.785594,4.346991e-08,Ours
6,ETTm1,336,0.424327,0.423381,0.222913,0.422129,0.421551,0.1,0.75,0.110815,0.714286,0.368539,13.147345,3.578755e-08,Ours
7,ETTm1,720,0.496095,0.488999,1.430322,0.461646,0.460045,0.1,0.75,0.095913,1.000000,0.427392,13.848778,1.632701e-08,Ours
8,Electricity,96,0.148275,0.146310,1.325576,0.239707,0.239500,0.0,1.00,0.031767,0.813084,0.134724,9.139225,4.570984e-09,Ours
9,Electricity,192,0.165101,0.164833,0.161767,0.256106,0.257408,0.0,1.00,0.043048,0.666667,0.153132,7.249021,1.342250e-08,Ours


,Dataset,Horizon,Comparison,MeanImprovement,CI_Low,CI_High,SignificantPositive,Significance
8,ETTh1,96,Direct-ShrinkAdaptive,-0.001978,-0.004286,0.000448,False,Not significant
10,ETTh1,192,Direct-ShrinkAdaptive,-0.006457,-0.009515,-0.002894,False,Direct significantly better
12,ETTh1,336,Direct-ShrinkAdaptive,-0.008738,-0.012672,-0.004596,False,Direct significantly better
14,ETTh1,720,Direct-ShrinkAdaptive,-0.029762,-0.035795,-0.023611,False,Direct significantly better
0,ETTm1,96,Direct-ShrinkAdaptive,-0.000380,-0.001116,0.000391,False,Not significant
2,ETTm1,192,Direct-ShrinkAdaptive,0.003760,0.001944,0.005723,True,Ours significantly better
4,ETTm1,336,Direct-ShrinkAdaptive,0.000946,-0.000936,0.002938,False,Not significant
6,ETTm1,720,Direct-ShrinkAdaptive,0.007096,0.003982,0.010414,True,Ours significantly better
24,Electricity,96,Direct-ShrinkAdaptive,0.001966,0.001538,0.002453,True,Ours significantly better
26,Electricity,192,Direct-ShrinkAdaptive,0.000267,-0.000084,0.000646,False,Not significant



# 21. Decision criteria

## 가장 강한 성공 시나리오

PatchTST에서 positive였던 Weather/Electricity뿐 아니라
iTransformer에서도 같은 regime이 반복됩니다.

예를 들어

$$
\mathrm{Weather/Electricity}
:
8/8\ \mathrm{wins}
$$

에 가까운 결과가 나오면

> historical memory complements strong forecasters across different backbone architectures

라는 주장이 매우 강해집니다.

---

## 부분 성공

Weather에서는 반복되지만 Electricity 또는 ETT에서는 혼합된 결과가 나올 수 있습니다.

이 경우에도 동일한 frozen memory module이
서로 다른 backbone에서 Weather를 반복적으로 개선한다면
PatchTST-specific artifact라는 설명은 약해집니다.

---

## 실패

iTransformer에서는 전반적으로 개선되지 않는다면
현재 method를 `model-agnostic augmentation`이라고 부르기 어렵습니다.

이 경우:

- PatchTST-specific interaction
- backbone calibration difference
- direct-error geometry difference
- opportunity identification failure

를 분석합니다.

그러나 Experiment 22 test 결과를 본 뒤
iTransformer용 gate/retriever를 별도로 test-tuning하지 않습니다.

---

## Oracle의 역할

Oracle headroom이 작으면
retrieval forecast 자체에 complementary signal이 거의 없는 것입니다.

Oracle headroom이 크지만 Ours가 개선하지 못하면

$$
\boxed{
\text{signal exists, but trust/opportunity identification fails}
}
$$

로 해석합니다.

---

# 이후 단계

Experiment 22가 성공하면 다음 우선순위는:

1. PatchTST + iTransformer 통합 backbone table
2. multi-seed confirmation
3. 필요 시 한 개의 newer strong backbone 추가

입니다.


## 22. Runtime summary

In [22]:

if len(
    summary_df
):
    runtime = (
        summary_df
        .groupby(
            "Dataset",
            as_index=False,
        )
        .agg(
            Conditions=(
                "Horizon",
                "size",
            ),
            TotalRuntimeMinutes=(
                "RuntimeMinutes",
                "sum",
            ),
            MeanRuntimeMinutes=(
                "RuntimeMinutes",
                "mean",
            ),
        )
    )

    runtime[
        "TotalRuntimeHours"
    ] = (
        runtime[
            "TotalRuntimeMinutes"
        ]
        / 60.0
    )

    display(
        runtime
    )

    print(
        "Recorded total runtime (hours):",
        summary_df[
            "RuntimeMinutes"
        ].sum()
        / 60.0,
    )


,Dataset,Conditions,TotalRuntimeMinutes,MeanRuntimeMinutes,TotalRuntimeHours
0,ETTh1,4,0.441558,0.110389,0.007359
1,ETTm1,4,2.959270,0.739818,0.049321
2,Electricity,4,29.886133,7.471533,0.498102
3,Weather,4,7.721903,1.930476,0.128698


Recorded total runtime (hours): 0.6834810678164164


## 23. Saved artifacts

In [23]:

print(
    "Experiment root:",
    ROOT,
)

for p in sorted(
    ROOT.rglob(
        "*"
    )
):
    if p.is_file():
        print(
            p.relative_to(
                ROOT
            )
        )


Experiment root: /data/dataset/strong_forecaster/itransformer_plus_frozen_historical_memory
artifacts/direct_parity.csv
artifacts/official_repo_commit.txt
bootstrap.csv
calibration/ETTh1_H192_scalar_alpha.csv
calibration/ETTh1_H192_shrink_lambda.csv
calibration/ETTh1_H336_scalar_alpha.csv
calibration/ETTh1_H336_shrink_lambda.csv
calibration/ETTh1_H720_scalar_alpha.csv
calibration/ETTh1_H720_shrink_lambda.csv
calibration/ETTh1_H96_scalar_alpha.csv
calibration/ETTh1_H96_shrink_lambda.csv
calibration/ETTm1_H192_scalar_alpha.csv
calibration/ETTm1_H192_shrink_lambda.csv
calibration/ETTm1_H336_scalar_alpha.csv
calibration/ETTm1_H336_shrink_lambda.csv
calibration/ETTm1_H720_scalar_alpha.csv
calibration/ETTm1_H720_shrink_lambda.csv
calibration/ETTm1_H96_scalar_alpha.csv
calibration/ETTm1_H96_shrink_lambda.csv
calibration/Electricity_H192_scalar_alpha.csv
calibration/Electricity_H192_shrink_lambda.csv
calibration/Electricity_H336_scalar_alpha.csv
calibration/Electricity_H336_shrink_lambda.csv
c